# Flock Energy — REFIT Washing-Machine Disaggregation

**Project repository:** `flock-refit-nilm`  
**Main task:** estimate washing-machine power from whole-house aggregate power at one-minute resolution.

In this project, I use the REFIT electricity dataset to work through the main stages of an appliance-disaggregation task: preparing the meter data, describing washing-machine usage, training and comparing models, and interpreting what the results could mean for an energy utility.

The target is the washing-machine power channel. The model only receives the household aggregate signal and features derived from that signal. Appliance sub-meter readings are used to create the target and check predictions; they are not model inputs.

I have kept the analysis in one notebook so the data decisions, implementation, intermediate tables, plots and final results can be reviewed in sequence. The notebook is intended to be rerun from a fresh Colab session. Numerical findings are generated from the supplied REFIT files, not typed into the report in advance.

## Running the notebook

1. Open the notebook in Google Colab.
2. Upload `REFIT_RAW_081116.7z`, `CLEAN_REFIT_081116.7z` and `CLEAN_READ_ME_081116.txt` to `MyDrive/flock-refit-archives/`.
3. Run the cells from top to bottom. Do not skip upstream phases if a later phase depends on their outputs.
4. The extracted CSVs are kept on Colab's local disk; processed Parquet files, metrics, figures and reports are saved to Google Drive.
5. A GPU is useful for the Seq2Point experiment. HistGradientBoosting runs on CPU.

**Reproducibility note:** the notebook starts with empty execution outputs. The results shown in the final submission should be produced by one fresh, sequential run.

## How I have structured the analysis

I have organised the work around the four sections in the assignment. The order matters: the modelling section depends on the channel mapping and data-quality decisions made earlier, and the business recommendation should use measured results rather than assumptions.

| Stage | What I do | Why it is needed |
|---|---|---|
| Raw-data preparation | Audit raw House 1, resolve timestamp duplicates, flag invalid readings, resample to one minute and record outages | The model should not learn from avoidable timestamp or measurement errors. |
| Washing-machine EDA | Use the official cleaned release, identify eligible homes, detect operating events and compare household usage | This establishes what the target signal looks like and how much it varies by household. |
| Model comparison | Compare two simple baselines, HistGradientBoosting and Seq2Point under the same LOHO folds | A model should be compared with a trivial reference and evaluated on households it did not train on. |
| Practical interpretation | Summarise observed usage and explain limitations and a possible utility pilot | Predictions are only useful when their limits and potential application are clear. |

### Why I compare these models

**HistGradientBoosting** is the tabular benchmark. It receives the current aggregate power, lagged aggregate values, rolling statistics and local calendar features. This gives a relatively efficient model that can learn nonlinear relationships without requiring a large neural network.

**Seq2Point** is the temporal model. It receives a 121-minute sequence of aggregate power and predicts the washing-machine power at the centre minute. This lets the CNN learn signal patterns directly rather than relying only on hand-designed summary features. Because the window is centred, it uses later aggregate readings as context; I therefore treat it as an offline/retrospective model, not a live causal model.

### Why I use Leave-One-Household-Out (LOHO)

A random row split would place measurements from the same home in both training and testing. That would make the test set too similar to the training data. Instead, I hold out one complete household at a time, train on the other selected homes, and repeat for each home. This tests whether the relationship learned from some households transfers to a household the model has not seen.

Seq2Point's early-stopping validation data is taken only from the five training households. The held-out household's appliance labels are reserved for final evaluation. All approaches are scored on the same common timestamps so their comparison is not affected by different test samples.

### What I consider a successful result

I do not judge disaggregation from overall MAE alone. Washing machines are inactive for most minutes, so an always-zero prediction can achieve a deceptively low MAE. I therefore read event-state F1, active/event-period power error, energy error, coverage and per-household variation together. A model may estimate power well during active periods but still trigger too often; the metrics need to show that trade-off.

## 0. Colab setup and storage

I keep the project outputs in a dedicated Drive folder so the figures and metric tables remain available after a Colab runtime stops. The large REFIT archives are also stored on Drive, but extracted CSVs are placed on Colab's local disk because repeatedly reading or extracting multi-gigabyte files from Drive is slower.

The processed one-minute Parquet files are stored separately. This avoids repeating the full-resolution ingestion work when I rerun later analysis cells. The notebook uses a versioned output folder so this run does not accidentally mix results with earlier experiments.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, sys, subprocess, json, logging, time, re, math, warnings

PROJECT_DIR = Path('/content/drive/MyDrive/flock-refit-nilm-final')
ARCHIVE_DIR = Path('/content/drive/MyDrive/flock-refit-archives')
DATA_DIR = Path('/content/refit_data_final')                 # large extracted CSVs: local Colab disk
RAW_DIR = DATA_DIR / 'raw'
CLEAN_DIR = DATA_DIR / 'clean'
RAW_ARCHIVE_DIR = DATA_DIR / 'raw_archives'
PROC_DIR = Path('/content/drive/MyDrive/flock-refit-processed/final_v7')
METRICS_DIR = PROJECT_DIR / 'outputs' / 'metrics'
FIG_DQ_DIR = PROJECT_DIR / 'outputs' / 'figures' / 'data_quality'
FIG_EDA_DIR = PROJECT_DIR / 'outputs' / 'figures' / 'eda'
FIG_MODEL_DIR = PROJECT_DIR / 'outputs' / 'figures' / 'modelling'
MODEL_DIR = PROJECT_DIR / 'models'

for p in [PROJECT_DIR, ARCHIVE_DIR, DATA_DIR, RAW_DIR, CLEAN_DIR, RAW_ARCHIVE_DIR,
          PROC_DIR, METRICS_DIR, FIG_DQ_DIR, FIG_EDA_DIR, FIG_MODEL_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_DIR)
print('Archive folder:', ARCHIVE_DIR)
print('Extracted raw/clean folders:', RAW_DIR, CLEAN_DIR)
print('Processed data folder:', PROC_DIR)
print('Free Colab disk (GB):', round(shutil.disk_usage('/content').free/1e9, 1))
print('Free Drive (GB):', round(shutil.disk_usage('/content/drive').free/1e9, 1))

## 1. Imports and dependencies

I use Pandas and NumPy for time-series handling, scikit-learn for the tabular benchmark, PyTorch for Seq2Point, and Matplotlib for the figures. `py7zr` is needed to extract the supplied archives and `pyarrow` is used to save the processed data in Parquet format.

The cell installs only packages that are missing. I do not upgrade NumPy or Pandas inside the notebook because Colab's preinstalled scientific stack is already integrated; changing core numerical packages mid-project can introduce compatibility problems that are unrelated to the analysis.

In [ ]:
import importlib.util
missing = [name for name in ['py7zr', 'pyarrow'] if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display, Image
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import precision_recall_fscore_support
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import py7zr
import pyarrow

print('Python:', sys.version.split()[0])
print('Pandas:', pd.__version__, '| NumPy:', np.__version__)
print('scikit-learn:', __import__('sklearn').__version__, '| PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2. Locate and extract the REFIT data

The assignment provides two archives and a README. The raw archive is used only for the House 1 cleaning task. The cleaned archive is used for the multi-household EDA and model comparison.

I check that the expected raw and cleaned CSV files are present before moving on. This is important because an incomplete extraction can otherwise look like a smaller dataset or silently remove a household from the analysis. The downloader is not assumed to work from Colab: the archives can be uploaded manually because the source host may reject scripted requests.

In [ ]:
RAW_ARCHIVE = ARCHIVE_DIR / 'REFIT_RAW_081116.7z'
CLEAN_ARCHIVE = ARCHIVE_DIR / 'CLEAN_REFIT_081116.7z'
README_SOURCE = ARCHIVE_DIR / 'CLEAN_READ_ME_081116.txt'


def csv_files(folder):
    return sorted(Path(folder).rglob('*.csv')) if Path(folder).exists() else []


def extract_if_needed(archive_path, target_dir, expected_min_csv, label):
    existing = csv_files(target_dir)
    if len(existing) >= expected_min_csv:
        print(f'{label}: extraction already present ({len(existing)} CSV files); skipping.')
        return
    if not archive_path.exists():
        raise FileNotFoundError(
            f'Missing {archive_path}. Upload {archive_path.name} to {ARCHIVE_DIR} and rerun this cell.'
        )
    print(f'Extracting {label}: {archive_path.name} ({archive_path.stat().st_size/1e9:.2f} GB)')
    with py7zr.SevenZipFile(archive_path, mode='r') as z:
        z.extractall(path=target_dir)
    found = csv_files(target_dir)
    print(f'{label}: extracted {len(found)} CSV files.')
    if len(found) < expected_min_csv:
        raise RuntimeError(f'{label}: expected at least {expected_min_csv} CSV files, found {len(found)}.')

extract_if_needed(CLEAN_ARCHIVE, CLEAN_DIR, expected_min_csv=20, label='Cleaned REFIT')
extract_if_needed(RAW_ARCHIVE, RAW_DIR, expected_min_csv=40, label='Raw REFIT')

if not README_SOURCE.exists():
    # Some archives also contain the README; search the extracted tree before failing.
    candidates = list(DATA_DIR.rglob('CLEAN_READ_ME_081116.txt'))
    if candidates:
        README_SOURCE = candidates[0]
if not README_SOURCE.exists():
    raise FileNotFoundError(f'Cleaned REFIT README not found. Upload CLEAN_READ_ME_081116.txt to {ARCHIVE_DIR}.')

print('Raw CSV count:', len(csv_files(RAW_DIR)))
print('Clean CSV count:', len(csv_files(CLEAN_DIR)))
print('README:', README_SOURCE)

## 3. Analysis settings

I keep the cohort preference, quality limits, event parameters and model settings together. This makes it possible to see which values are fixed by the analysis and which are derived from the data.

The preferred cohort is Houses 1, 2, 5, 6, 7 and 8. The selection code verifies the official appliance mapping and the actual files before accepting a home. A household is not accepted merely because its file exists: it must have a dedicated `Washing Machine` channel and enough jointly valid data and complete events for the planned comparison.

The 20 W / 27-minute ON / 13-minute OFF event settings are taken from a published REFIT-specific study and are treated as an operational definition for this analysis, not as universal washing-machine physics. The 240-minute rule is a separate project quality-control flag for unusually long events.

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s — %(message)s', force=True)
logger = logging.getLogger('flock_refit_notebook')

SOLAR_HOUSES = {3, 11, 21}
PREFERRED_HOUSES = [1, 2, 5, 6, 7, 8]
MIN_VALID_DAYS = 30
MIN_COMPLETE_CYCLES = 10

AGG_PHYSICAL_BOUND_W = 23_000
IAM_RATED_LIMIT_W = 4_000
ZERO_FILL_MIN_CONSECUTIVE = 2
FORWARD_FILL_MIN_CONSECUTIVE = 10

DEFAULT_THRESHOLD_W = 20.0
MIN_ON_DURATION_MIN = 27
MIN_OFF_DURATION_MIN = 13
MAX_CYCLE_DURATION_MIN = 240  # data-quality flag only; not a REFIT literature parameter

MAX_HGBR_ROWS_PER_HOUSE = 200_000
RANDOM_SEED = 42

SEQ_WINDOW_MINUTES = 121
SEQ_MAX_WINDOWS_PER_HOUSE = 25_000
SEQ_VAL_MAX_WINDOWS_PER_HOUSE = 5_000
SEQ_EPOCHS = 15
SEQ_PATIENCE = 4
SEQ_VAL_FRACTION = 0.15
SEQ_BATCH_SIZE = 512
SEQ_ACTIVE_LOSS_WEIGHT = 4.0
SEQ_TARGET_SCALE_W = 2_000.0

ILLUSTRATIVE_WINDOWS = {
    'primary_16_20': (16, 20),
    'sensitivity_17_21': (17, 21),
}

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
# Make CUDA convolution behaviour reproducible where supported; benchmark mode can
# choose different kernels between runs and is not needed for this small fixed model.
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print('Configuration loaded.')
print('Primary activation rule:', DEFAULT_THRESHOLD_W, 'W; min ON', MIN_ON_DURATION_MIN,
      'min; merge OFF gaps <', MIN_OFF_DURATION_MIN, 'min')
print('Models: HistGradientBoosting + Seq2Point CNN')
print('Seq2Point:', SEQ_WINDOW_MINUTES, 'min window, up to', SEQ_EPOCHS,
      'epochs with', SEQ_PATIENCE, 'epoch early-stopping patience')

## 4. Data-ingestion and quality-control functions

I use the Unix timestamp as the canonical time axis and keep it in UTC through sorting, duplicate handling and resampling. I convert timestamps to `Europe/London` only when calculating local hour-of-day and day-of-week patterns. This avoids treating the daylight-saving clock change as an ordinary duplicate or outage.

The functions below are shared by the raw House 1 pipeline and each selected cleaned household. Reusing the same resampling and missing-data logic makes the household comparisons more consistent. The code also records quality flags rather than hiding every unusual value by deleting it.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from numpy.lib.stride_tricks import sliding_window_view

### 4.1 Read files, parse timestamps and audit the raw measurements

These functions load the source files in chunks where needed, identify the timestamp unit from its magnitude, compare the local-time field with the Unix timestamp, and calculate per-channel data-quality summaries.

In [ ]:
"""
Shared ingestion, deduplication and 1-minute resampling for REFIT data.
Used for raw House 1 and every selected cleaned household.
"""
import logging
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Physical plausibility bound: UK single-phase 100A x 230V
AGG_PHYSICAL_BOUND_W = 23_000
# CurrentCost IAM documented rated limit
IAM_RATED_LIMIT_W = 4_000
# Minimum consecutive zero-minutes to flag as suspected provider zero-fill
ZERO_FILL_MIN_CONSECUTIVE = 2
# Minimum consecutive identical-value minutes to flag as suspected forward-fill
FORWARD_FILL_MIN_CONSECUTIVE = 10


# ---------------------------------------------------------------------------
# 1. Load raw CSV
# ---------------------------------------------------------------------------

def load_refit_csv(path, chunksize: int = 500_000) -> pd.DataFrame:
    """Load one REFIT CSV, or several parts of one house (concatenated in the order given)."""
    paths = [Path(p) for p in (path if isinstance(path, (list, tuple)) else [path])]
    chunks = []
    for pth in paths:
        n0 = sum(len(c) for c in chunks)
        for chunk in pd.read_csv(pth, chunksize=chunksize, low_memory=False):
            chunks.append(chunk)
        logger.info("Loaded %s: %d rows", pth.name, sum(len(c) for c in chunks) - n0)
    df = pd.concat(chunks, ignore_index=True)
    logger.info("Total %d rows from %d file(s); columns: %s", len(df), len(paths), list(df.columns))
    return df


def parse_timestamps(df: pd.DataFrame, log_path: Optional[Path] = None) -> pd.DataFrame:
    """
    Parse Unix timestamp column to UTC datetime.
    - Auto-detects seconds vs milliseconds.
    - Adds 'utc' column (UTC datetime) and 'local' column (Europe/London).
    - Compares with DateTime string column if present.
    """
    df = df.copy()

    # Find unix column (case-insensitive)
    unix_col = next((c for c in df.columns if "unix" in c.lower()), None)
    if unix_col is None:
        raise ValueError(f"No Unix timestamp column found. Columns: {list(df.columns)}")

    vals = df[unix_col].dropna()
    # Detect unit: Unix seconds are ~1e9, milliseconds ~1e12
    median_val = float(vals.median())
    unit = "ms" if median_val > 1e11 else "s"
    logger.info("Unix timestamp unit detected: %s (median value=%.0f)", unit, median_val)

    df["utc"] = pd.to_datetime(df[unix_col], unit=unit, utc=True, errors="coerce")
    df["local"] = df["utc"].dt.tz_convert("Europe/London")

    parse_failures = df["utc"].isna().sum()
    if parse_failures > 0:
        logger.warning("%d timestamp parse failures", parse_failures)

    # Compare with DateTime string column if present
    dt_col = next((c for c in df.columns if c.lower() in ("datetime", "date_time", "time")), None)
    if dt_col is not None:
        naive = pd.to_datetime(df[dt_col], errors="coerce")  # ISO strings; dayfirst would corrupt them
        df["datetime_as_utc"] = naive.dt.tz_localize("UTC")
        df["datetime_str"] = naive.dt.tz_localize("Europe/London", ambiguous="NaT", nonexistent="NaT")
        logger.info("Time column compared with Unix timestamp as UTC and as London local time")

    return df


# ---------------------------------------------------------------------------
# 1b. Audit (run on the parsed-but-unsorted raw frame)
# ---------------------------------------------------------------------------

def dst_days(first_utc: pd.Timestamp, last_utc: pd.Timestamp) -> pd.DataFrame:
    """Local calendar days that are not 24 h long (BST clock changes)."""
    days = pd.date_range(first_utc.tz_convert("Europe/London").normalize().tz_localize(None),
                         last_utc.tz_convert("Europe/London").normalize().tz_localize(None),
                         freq="D")
    loc = days.tz_localize("Europe/London", ambiguous="NaT", nonexistent="shift_forward")
    nxt = (days + pd.Timedelta(days=1)).tz_localize("Europe/London", ambiguous="NaT",
                                                     nonexistent="shift_forward")
    hrs = (nxt - loc).total_seconds() / 3600
    out = pd.DataFrame({"local_date": days.date, "hours": hrs})
    return out[out["hours"].round(0) != 24].reset_index(drop=True)


def audit_raw(df: pd.DataFrame, power_cols: list[str], agg_col: str) -> tuple[pd.DataFrame, dict]:
    """
    Basic inspection of a parsed (not yet sorted/deduplicated) frame.
    Returns (per-channel table, scalar summary dict).
    """
    rows = []
    for col in power_cols:
        v = pd.to_numeric(df[col], errors="coerce")
        rows.append({
            "channel": col, "dtype": str(df[col].dtype), "rows": int(len(v)),
            "missing": int(v.isna().sum()), "negative": int((v < 0).sum()),
            "above_4000W": int((v > IAM_RATED_LIMIT_W).sum()),
            "above_23000W": int((v > AGG_PHYSICAL_BOUND_W).sum()),
            "min": float(v.min()), "median": float(v.median()),
            "p99": float(v.quantile(0.99)), "p99_9": float(v.quantile(0.999)),
            "max": float(v.max()),
        })
    ch = pd.DataFrame(rows)

    utc = df["utc"]
    srt = utc.sort_values()
    diffs = srt.drop_duplicates().diff().dt.total_seconds().dropna()
    diffs = diffs[diffs > 0]
    first, last = srt.iloc[0], srt.iloc[-1]
    dst = dst_days(first, last)
    summary = {
        "n_rows": int(len(df)), "n_columns": int(df.shape[1]),
        "columns": list(map(str, df.columns)),
        "first_utc": str(first), "last_utc": str(last),
        "timestamp_parse_failures": int(utc.isna().sum()),
        "out_of_order_rows": int((utc.diff().dt.total_seconds() < 0).sum()),
        "exact_duplicate_rows": int(df.drop(columns=[c for c in ["local", "datetime_str", "datetime_as_utc"] if c in df.columns]).duplicated().sum()),
        "duplicate_timestamps": int(utc.duplicated().sum()),
        "interval_median_s": float(diffs.median()),
        "interval_p1_s": float(diffs.quantile(0.01)),
        "interval_p99_s": float(diffs.quantile(0.99)),
        "intervals_gt_60s": int((diffs > 60).sum()),
        "intervals_gt_10min": int((diffs > 600).sum()),
        "intervals_gt_1h": int((diffs > 3600).sum()),
        "dst_clock_change_days": [f"{d} ({h:.0f}h)" for d, h in zip(dst["local_date"], dst["hours"])],
    }
    if "datetime_as_utc" in df.columns:
        d_utc = (df["utc"] - df["datetime_as_utc"]).abs().dt.total_seconds()
        ok = d_utc.notna()
        summary["time_col_equals_unix_as_utc_pct"] = round(float((d_utc[ok] <= 1).mean() * 100), 3)
        summary["time_col_max_abs_diff_vs_utc_s"] = float(d_utc.max())
        d_loc = (df["utc"] - df["datetime_str"].dt.tz_convert("UTC")).abs().dt.total_seconds()
        ok2 = d_loc.notna()
        summary["time_col_equals_unix_as_london_local_pct"] = round(float((d_loc[ok2] <= 1).mean() * 100), 3)
        summary["time_col_interpretation"] = (
            "UTC" if summary["time_col_equals_unix_as_utc_pct"] > 99 else
            "London local time" if summary["time_col_equals_unix_as_london_local_pct"] > 99 else "mixed/unclear")
    return ch, summary


### 4.2 Resolve duplicate timestamps and flag invalid readings

Exact duplicates are removed. Conflicting measurements at the same timestamp are reduced with a per-channel median; the issue flag uses the conservative value if any duplicate row is flagged. Negative appliance readings and documented invalid IAM values are recorded rather than silently interpreted as zero.

In [ ]:
# ---------------------------------------------------------------------------
# 2. Deduplication
# ---------------------------------------------------------------------------

def deduplicate(df: pd.DataFrame, power_cols: list[str],
                log_path: Optional[Path] = None) -> tuple[pd.DataFrame, dict]:
    """
    Fixed duplicate policy:
    1. Drop exact duplicate rows.
    2. For conflicting rows at the same Unix timestamp, keep per-channel median
       (conservative max for a binary Issues flag).
    3. Log counts and up to 50 sample timestamps per type.
    Returns (clean_df, stats).
    """
    helper = [c for c in ["local", "datetime_str", "datetime_as_utc"] if c in df.columns]
    exact_mask = df.drop(columns=helper).duplicated(keep="first")
    exact_ts = df.loc[exact_mask, "utc"]
    df = df.loc[~exact_mask]

    dup_mask = df["utc"].duplicated(keep=False)
    conflict_ts = df.loc[dup_mask, "utc"].drop_duplicates()
    n_conflict_rows = int(dup_mask.sum())

    if n_conflict_rows > 0:
        agg = {c: "median" for c in power_cols if c in df.columns}
        for c in df.columns:
            if "issue" in c.lower():
                agg[c] = "max"
        first_vals = df[dup_mask].groupby("utc").first()
        collapsed = df[dup_mask].groupby("utc").agg(agg)
        for c in first_vals.columns:
            if c not in collapsed.columns:
                collapsed[c] = first_vals[c]
        collapsed = collapsed.reset_index()
        df = pd.concat([df[~dup_mask], collapsed[df.columns]], ignore_index=True)

    df = df.sort_values("utc").reset_index(drop=True)
    assert df["utc"].is_unique, "Timestamps not unique after deduplication"
    assert df["utc"].is_monotonic_increasing

    stats = {"exact_duplicates_dropped": int(exact_mask.sum()),
             "conflicting_rows_collapsed": n_conflict_rows,
             "conflicting_timestamps": int(len(conflict_ts)),
             "rows_after": int(len(df))}
    logger.info("Deduplication: %s", stats)

    if log_path is not None:
        rows = ([{"utc": t, "type": "exact"} for t in exact_ts.head(50)] +
                [{"utc": t, "type": "conflicting"} for t in conflict_ts.head(50)])
        pd.DataFrame(rows, columns=["utc", "type"]).to_csv(log_path, index=False)
    return df, stats


# ---------------------------------------------------------------------------
# 3. Invalid and extreme values
# ---------------------------------------------------------------------------

def flag_invalid_values(df: pd.DataFrame, power_cols: list[str],
                        agg_col: str, is_solar_house: bool = False,
                        log_path: Optional[Path] = None) -> pd.DataFrame:
    """
    Apply fixed invalid-value rules. Returns df with values set to NaN where invalid.
    Logs modified values to CSV.
    """
    df = df.copy()
    log_records = []

    for col in power_cols:
        if col not in df.columns:
            continue
        is_agg = (col == agg_col)

        # Coerce power columns to numeric and remove +/- infinity explicitly.
        # NaN is retained as missing; non-numeric tokens are also logged as invalid.
        original = df[col]
        numeric = pd.to_numeric(original, errors="coerce")
        nonnumeric_mask = original.notna() & numeric.isna()
        inf_mask = pd.Series(np.isinf(numeric.to_numpy(dtype=float, na_value=np.nan)), index=df.index)
        invalid_parse_mask = nonnumeric_mask | inf_mask
        if invalid_parse_mask.any():
            log_records += [{"utc": t, "channel": col, "value": v, "reason": "non_numeric_or_infinite"}
                            for t, v in zip(df.loc[invalid_parse_mask, "utc"], original.loc[invalid_parse_mask])]
        df[col] = numeric.mask(inf_mask, np.nan)

        # Negative power
        neg_mask = df[col] < 0
        if neg_mask.any():
            if is_agg and is_solar_house:
                logger.info("%s: %d negative readings kept (solar-affected aggregate)", col, neg_mask.sum())
            else:
                log_records += [{"utc": t, "channel": col, "value": v, "reason": "negative"}
                                 for t, v in zip(df.loc[neg_mask, "utc"], df.loc[neg_mask, col])]
                df.loc[neg_mask, col] = np.nan

        # IAM above 4000 W (appliance channels only, raw data)
        if not is_agg:
            iam_mask = df[col] > IAM_RATED_LIMIT_W
            if iam_mask.any():
                log_records += [{"utc": t, "channel": col, "value": v, "reason": "above_IAM_4000W"}
                                 for t, v in zip(df.loc[iam_mask, "utc"], df.loc[iam_mask, col])]
                df.loc[iam_mask, col] = np.nan

        # Aggregate physical bound alert
        if is_agg:
            phys_mask = df[col] > AGG_PHYSICAL_BOUND_W
            if phys_mask.any():
                logger.warning("%s: %d values above physical bound %.0f W (flagged, not removed)",
                               col, phys_mask.sum(), AGG_PHYSICAL_BOUND_W)
                for t, v in zip(df.loc[phys_mask, "utc"], df.loc[phys_mask, col]):
                    log_records.append({"utc": t, "channel": col, "value": v, "reason": "above_physical_bound_alert"})

        # 99.9th percentile report
        p999 = df[col].quantile(0.999)
        above = (df[col] > p999).sum()
        if above > 0:
            logger.info("%s p99.9=%.1f W, %d values above (retained)", col, p999, above)

    # Sub-meter > aggregate diagnostic (do not remove, sensors are async)
    appliance_cols = [c for c in power_cols if c != agg_col]
    if appliance_cols:
        sub_sum = df[appliance_cols].sum(axis=1, min_count=1)
        discrepancy = (sub_sum > df[agg_col] * 1.1) & df[agg_col].notna() & sub_sum.notna()
        logger.info("Sub-meter > aggregate (>10%%) diagnostic: %d rows", discrepancy.sum())

    if log_path is not None and log_records:
        pd.DataFrame(log_records).to_csv(log_path, index=False)
        logger.info("Modified values log saved: %s (%d entries)", log_path, len(log_records))

    return df


### 4.3 Resample to one minute and handle missing runs

The source interval is calculated from the observed timestamps. A complete one-minute UTC index is built, and a minute is valid when at least half of the expected source readings are present. Short internal gaps are interpolated only when the entire missing run is no longer than two minutes. The remaining missing runs are written to the outage table.

In [ ]:
# ---------------------------------------------------------------------------
# 4. 1-minute resampling
# ---------------------------------------------------------------------------

def resample_to_1min(df: pd.DataFrame, power_cols: list[str]) -> pd.DataFrame:
    """
    Resample to 1-minute UTC bins with the 50% coverage primary rule.
    Returns a complete-index DataFrame with NaN for under-covered bins.
    """
    df = df.set_index("utc").sort_index()
    df = df[power_cols]

    # Estimate source sampling interval
    idx_sorted = df.index.drop_duplicates().sort_values()
    diffs = idx_sorted.to_series().diff().dt.total_seconds().dropna()
    diffs = diffs[diffs > 0]
    median_interval = float(diffs.median())
    expected_per_min = 60.0 / median_interval
    logger.info("Source median interval: %.1f s, expected readings/min: %.1f",
                median_interval, expected_per_min)

    thresh_50 = int(np.ceil(0.5 * expected_per_min))
    thresh_80 = int(np.ceil(0.8 * expected_per_min))

    # Build complete 1-minute index
    start = df.index.min().floor("min")
    # Resample labels are bin-start timestamps. The last observed source sample
    # belongs to its floored minute; using ceil() would append a false empty bin
    # whenever the last timestamp is not exactly on a minute boundary.
    end = df.index.max().floor("min")
    if end < start:
        raise ValueError("Resampling produced an invalid time range.")
    full_idx = pd.date_range(start, end, freq="1min", tz="UTC")

    results = {}
    coverage_rows = []
    for col in power_cols:
        s = df[col].dropna()
        # Resample: count valid readings and mean value
        count_1min = s.resample("1min").count().reindex(full_idx, fill_value=0)
        mean_1min = s.resample("1min").mean().reindex(full_idx)

        valid_50 = count_1min >= thresh_50
        valid_80 = count_1min >= thresh_80
        valid_1  = count_1min >= 1

        # Primary: 50% rule
        out = mean_1min.copy()
        out[~valid_50] = np.nan
        results[col] = out
        results[f"{col}_valid_count"] = count_1min

        total = len(full_idx)
        coverage_rows.append({
            "channel": col,
            "total_minutes": total,
            "valid_ge1_pct": round(valid_1.sum() / total * 100, 2),
            "valid_50pct_pct": round(valid_50.sum() / total * 100, 2),
            "valid_80pct_pct": round(valid_80.sum() / total * 100, 2),
            "median_interval_s": round(median_interval, 2),
            "expected_per_min": round(expected_per_min, 2),
        })

    out_df = pd.DataFrame(results, index=full_idx)
    out_df.index.name = "utc"
    coverage_df = pd.DataFrame(coverage_rows)

    logger.info("Resampled to 1-minute: %d bins, %s to %s", len(full_idx), start, end)
    return out_df, coverage_df, median_interval


# ---------------------------------------------------------------------------
# 5. Short-gap interpolation
# ---------------------------------------------------------------------------

def interpolate_short_gaps(df: pd.DataFrame, power_cols: list[str],
                           target_col: Optional[str] = None,
                           max_gap: int = 2) -> pd.DataFrame:
    """
    Interpolate only complete, internal missing runs of 1..max_gap minutes.

    Safeguards:
    - Never interpolate a target_col.
    - Never interpolate leading/trailing gaps.
    - Never partially fill a longer gap.
    - Never refill any minute belonging to an aggregate-based provider-fill run.
      If one minute in a missing run is provider-flagged, protect the whole run.
    - The imputed flag is True only when a numeric value was actually filled.
    """
    df = df.copy()
    suspect_cols = [c for c in df.columns if c.endswith("_suspected_fill")]
    if suspect_cols:
        protected = df[suspect_cols].fillna(False).astype(bool).any(axis=1)
    else:
        protected = pd.Series(False, index=df.index)

    for col in power_cols:
        if col not in df.columns:
            continue

        s = pd.to_numeric(df[col], errors="coerce").copy()
        missing = s.isna()

        if col == target_col:
            df[f"{col}_orig_missing"] = missing
            df[f"{col}_interpolated"] = False
            df[f"{col}_final_missing"] = missing
            continue

        # Every missing run receives one run ID. The mask below applies to the
        # entire run; a provider-protected minute prevents partial filling.
        run_id = s.notna().cumsum()
        gap_size = missing.groupby(run_id).transform("sum")
        protected_run = protected.groupby(run_id).transform("any")

        candidate = s.interpolate(method="time", limit_area="inside")
        fillable = (missing & (gap_size <= max_gap) & candidate.notna()
                    & ~protected_run)

        s.loc[fillable] = candidate.loc[fillable]
        actually_filled = fillable & s.notna()

        df[col] = s
        df[f"{col}_orig_missing"] = missing
        df[f"{col}_interpolated"] = actually_filled
        df[f"{col}_final_missing"] = s.isna()

        logger.info("Short-gap interpolation %s: %d bins filled; %d protected provider-fill bins",
                    col, int(actually_filled.sum()), int((missing & protected).sum()))

    return df

# ---------------------------------------------------------------------------
# 6. Outage detection
# ---------------------------------------------------------------------------

def detect_outages(df: pd.DataFrame, power_cols: list[str],
                   min_gap_min: int = 3) -> pd.DataFrame:
    """
    Find runs of missing values >= min_gap_min minutes.
    Returns DataFrame with channel, start, end, duration_min.
    """
    records = []
    for col in power_cols:
        if col not in df.columns:
            continue
        m = df[col].isna().to_numpy().astype(np.int8)
        if not m.any():
            continue
        edges = np.diff(np.concatenate(([0], m, [0])))
        starts = np.where(edges == 1)[0]
        ends = np.where(edges == -1)[0] - 1
        for st, en in zip(starts, ends):
            dur = int(en - st + 1)
            if dur >= min_gap_min:
                records.append({"channel": col,
                                "start_utc": df.index[st],
                                "end_utc": df.index[en],
                                "duration_min": dur})
    outages = pd.DataFrame(records)
    if not outages.empty:
        outages = outages.sort_values("duration_min", ascending=False).reset_index(drop=True)
    return outages


### 4.4 Flag possible provider-filled aggregate intervals

The repeated-zero and long-constant-value rules are diagnostics for possible provider filling. They are not proof that every flagged interval is an outage. The mask is retained so protected intervals are not accidentally refilled during interpolation.

In [ ]:
# ---------------------------------------------------------------------------
# 7. Provider fill detection (cleaned data only)
# ---------------------------------------------------------------------------

def flag_suspected_provider_fill(df: pd.DataFrame, agg_col: str,
                                 other_cols: Optional[list] = None) -> pd.DataFrame:
    """
    Flag aggregate patterns that may be provider-zero-filled or forward-filled.
    All flagged minutes are set to NaN in aggregate and appliance channels. The
    interpolation function explicitly protects these rows from being refilled.

    Heuristics: exact 0 W for >=2 consecutive minutes, or an identical non-null
    aggregate value for >=10 consecutive one-minute bins. These are diagnostics,
    not proof that every flagged interval is an outage.
    """
    if agg_col not in df.columns:
        return df
    df = df.copy()
    s = pd.to_numeric(df[agg_col], errors="coerce")

    is_zero = s.eq(0.0).fillna(False)
    zero_run_id = (~is_zero).cumsum()
    zero_run_len = is_zero.groupby(zero_run_id).transform("sum")
    zero_fill_mask = is_zero & (zero_run_len >= ZERO_FILL_MIN_CONSECUTIVE)

    valid = s.notna()
    same_as_previous = s.eq(s.shift(1)) & valid
    run_start = ~same_as_previous
    value_run_id = run_start.cumsum()
    value_run_len = valid.groupby(value_run_id).transform("sum")
    forward_fill_mask = valid & (value_run_len >= FORWARD_FILL_MIN_CONSECUTIVE)

    suspected = zero_fill_mask | forward_fill_mask
    df[f"{agg_col}_suspected_fill"] = suspected
    logger.info("Suspected provider fill: %d zero-fill mins, %d constant-run mins",
                int(zero_fill_mask.sum()), int(forward_fill_mask.sum()))

    for c in [agg_col] + [c for c in (other_cols or []) if c in df.columns]:
        df.loc[suspected, c] = np.nan
    return df


## 5. Appliance mapping and household selection functions

The REFIT README maps appliance labels to channel numbers separately for each house. I parse that mapping instead of assuming, for example, that `Appliance2` always means the washing machine. I also keep `Washing Machine` separate from `Washer Dryer`; combining those labels would change the prediction target.

The `Issues` field is used only after checking the supplied README and CSV schema. Issue-flagged target minutes are excluded from regression fitting and scoring, but they remain available when constructing event continuity. This avoids turning a measurement-consistency flag into an artificial OFF period that could split a real operating event.

### 5.1 Parse the official appliance mapping

The README parser handles the supplied mapping layouts and checks the resulting house/channel labels against the actual CSV columns. This prevents a channel number from being mistaken for the same appliance in every household.

In [ ]:
"""
README parsing, household selection, and full per-house cleaning pipeline.
"""
import logging, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

SOLAR_HOUSES = {3, 11, 21}
PREFERRED_HOUSES = [1, 2, 5, 6, 7, 8]
MIN_VALID_DAYS = 30
MIN_COMPLETE_CYCLES = 10  # verified in Section 2 after cycle detection


# ---------------------------------------------------------------------------
# README parsing
# ---------------------------------------------------------------------------

KNOWN_APPLIANCE_WORDS = ("fridge", "freezer", "washing machine", "washer dryer", "dishwasher",
                         "television", "tv", "kettle", "microwave", "computer", "tumble dryer",
                         "toaster", "heater", "hi-fi", "router", "oven", "cooker", "lamp")


def _parse_house_block(block: str) -> list[tuple[int, str]]:
    """Return [(column_index, label)] from one house's README block. Tries two layouts."""
    found = []
    # Layout A: one mapping per line  ("Appliance5 = Washing Machine", "5: Washing Machine", "5. Washing Machine")
    for line in block.splitlines():
        m = re.match(r"\s*(?:appliance\s*)?(\d{1,2})\s*[=:.\-)]\s*([A-Za-z][^\n]*)$", line, re.IGNORECASE)
        if m and 1 <= int(m.group(1)) <= 9:
            found.append((int(m.group(1)), m.group(2).strip().rstrip(",;.")))
    if found:
        return found
    # Layout B: a single delimited list of appliance names, in channel order
    for line in block.splitlines():
        if sum(w in line.lower() for w in KNOWN_APPLIANCE_WORDS) >= 2:
            items = [i.strip(" .\t") for i in re.split(r"[,;|\t]|\s-\s", line) if i.strip(" .\t")]
            items = [re.sub(r"^(house\s*\d+\s*[:\-]?\s*)", "", i, flags=re.IGNORECASE) for i in items]
            items = [i for i in items if i]
            if items and items[0].lower().startswith("aggregate"):
                items = items[1:]
            return [(i + 1, lab) for i, lab in enumerate(items[:9])]
    return []


def channel_note(label: str) -> str:
    """Flag README wording such as '(Change 20 May 2014)', '(Changes 25 Mar 2015)', 'replaced', 'solar'."""
    if re.search(r"\bchange[sd]?\b|\breplaced?\b|\bsolar\b|\bpv\b", label, re.IGNORECASE):
        return "channel_change_or_solar"
    return ""


def parse_readme(readme_path: Path) -> tuple[pd.DataFrame, dict]:
    """
    Parse the CLEAN REFIT README to extract:
    - appliance_map: DataFrame(house, column_index, appliance_label, notes)
    - issues_info: dict describing whether/how the Issues field is documented
    Raises a clear error (with the README head) if too little could be parsed, so the
    problem is caught at Phase 0 instead of silently producing an empty mapping.
    """
    text = readme_path.read_text(errors="replace")
    records = []
    parts = re.split(r"(?im)^\s*house\s+(\d+)\b", text)
    for i in range(1, len(parts), 2):
        house_num = int(parts[i])
        block = parts[i + 1] if i + 1 < len(parts) else ""
        for col_idx, label in _parse_house_block(block):
            notes = channel_note(label)
            records.append({"house": house_num, "column_index": col_idx,
                            "appliance_label": label, "notes": notes})

    appliance_map = pd.DataFrame(records, columns=["house", "column_index", "appliance_label", "notes"])
    n_houses = appliance_map["house"].nunique() if len(appliance_map) else 0
    n_wm = int(appliance_map["appliance_label"].str.lower().str.contains("washing machine").sum()) \
        if len(appliance_map) else 0
    logger.info("README parsed: %d mappings, %d houses, %d 'Washing Machine' channels",
                len(appliance_map), n_houses, n_wm)
    if n_houses < 4 or n_wm < 4:
        head = "\n".join(text.splitlines()[:80])
        raise RuntimeError(
            f"README parsing looks wrong (houses={n_houses}, washing-machine channels={n_wm}).\n"
            f"Paste the output below to get the parser fixed:\n----- README (first 80 lines) -----\n{head}")

    issues_info = {}
    if re.search(r"\bissues?\b", text, re.IGNORECASE):
        issues_info["documented"] = True
        issues_info["raw_context"] = re.findall(r".{0,120}issues?.{0,120}", text, re.IGNORECASE)[:3]
        logger.info("'Issues' mentioned in README: %s", issues_info["raw_context"])
    else:
        issues_info["documented"] = False
        logger.info("No 'Issues' field mentioned in README.")
    return appliance_map, issues_info


def find_appliance_col(df: pd.DataFrame, house: int, appliance_map: pd.DataFrame,
                       target_labels: list[str]) -> Optional[str]:
    """
    Column name of the target appliance in a house. Matches the README channel number to the
    column literally named Appliance{N}; falls back to positional order only if absent.
    'Washer Dryer' never matches a 'washing machine' label.
    """
    house_map = appliance_map[appliance_map["house"] == house]
    cols_by_num = {}
    for c in df.columns:
        m = re.fullmatch(r"\s*appliance\s*(\d+)\s*", str(c), re.IGNORECASE)
        if m:
            cols_by_num[int(m.group(1))] = c
    for _, row in house_map.iterrows():
        label = str(row["appliance_label"]).lower()
        if any(t.lower() in label for t in target_labels):
            idx = int(row["column_index"])
            if idx in cols_by_num:
                return cols_by_num[idx]
    return None


def find_aggregate_col(df: pd.DataFrame) -> str:
    for c in df.columns:
        if "aggregate" in c.lower():
            return c
    raise ValueError(f"No aggregate column found. Columns: {list(df.columns)}")


### 5.2 Apply the household eligibility rule

The selection function checks the dedicated washing-machine label, solar-house exclusions and documented channel changes. The preferred cohort is only a starting list; acceptance depends on the checks recorded in the selection table.

In [ ]:
# ---------------------------------------------------------------------------
# Household selection
# ---------------------------------------------------------------------------

def select_households(clean_dir: Path, appliance_map: pd.DataFrame,
                      out_csv: Path, target: int = 6) -> list[int]:
    """
    Select eligible households following the fixed rule:
    1. Dedicated 'Washing Machine' label (not 'Washer Dryer').
    2. Not solar-affected (houses 3, 11, 21).
    3. No channel_change_or_solar note.
    Returns list of selected house IDs.
    """
    WM_LABEL = ["washing machine"]
    WD_LABEL = ["washer dryer"]

    records = []
    house_files = {h: find_house_csv(clean_dir, h) for h in range(1, 22)}
    house_files = {h: p for h, p in house_files.items() if p is not None}

    for house in sorted(house_files):
        reason = None
        house_map = appliance_map[appliance_map["house"] == house]
        labels = house_map["appliance_label"].str.lower()
        wm_rows = house_map[labels.str.contains("washing machine")]
        has_wd = bool(labels.str.contains("washer dryer").any())

        if house in SOLAR_HOUSES:
            reason = "solar-affected aggregate (Houses 3, 11, 21)"
        elif len(wm_rows) == 0:
            reason = "no Washing Machine channel (Washer Dryer is not used as a substitute)"
        elif len(wm_rows) > 1:
            reason = "two separate washing-machine channels (summing not implemented; excluded)"
        elif (wm_rows["notes"] == "channel_change_or_solar").any():
            reason = "README notes a change on the washing-machine channel"

        records.append({
            "house": house, "included": reason is None, "reason": reason or "eligible",
            "wm_channel": f"Appliance{int(wm_rows['column_index'].iloc[0])}" if len(wm_rows) == 1 else "",
            "wm_label": wm_rows["appliance_label"].iloc[0] if len(wm_rows) == 1 else "",
            "also_has_washer_dryer": has_wd,
        })

    sel_df = pd.DataFrame(records)

    # Prefer the PREFERRED_HOUSES list, then fill from remaining eligible
    eligible = sel_df[sel_df["included"]]["house"].tolist()
    preferred = [h for h in PREFERRED_HOUSES if h in eligible]
    others = [h for h in eligible if h not in preferred]
    selected = (preferred + others)[:target]

    sel_df["selected"] = sel_df["house"].isin(selected)
    sel_df.to_csv(out_csv, index=False)
    logger.info("Household selection: %d eligible, %d selected: %s", len(eligible), len(selected), selected)
    return selected


# ---------------------------------------------------------------------------
# Full 1-minute cleaning pipeline (one house)
# ---------------------------------------------------------------------------


### 5.3 Clean one household at a time

The household cleaning function applies the shared timestamp, duplicate, invalid-value, resampling, provider-fill, interpolation and outage logic. Processing one house at a time keeps memory use manageable and saves a separate one-minute Parquet file for later phases.

In [ ]:
def clean_house(csv_path, house: int, appliance_map: pd.DataFrame,
                issues_info: dict, out_dir: Path, metrics_dir: Path,
                is_raw: bool = False, is_solar: bool = False) -> Optional[pd.DataFrame]:
    """
    Full cleaning pipeline for one house:
    1. Load CSV
    2. Parse timestamps
    3. Deduplicate
    4. Flag invalid values
    5. Resample to 1 min (50% rule)
    6. Provider fill detection (cleaned data only)
    7. Short-gap interpolation (not on target)
    8. Detect outages
    Returns 1-minute DataFrame, saved to Parquet.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)

    pfx = "raw_" if is_raw else ""          # keeps Section 1 (raw) files distinct from cleaned-run files
    names = [p.name for p in (csv_path if isinstance(csv_path, (list, tuple)) else [csv_path])]
    logger.info("=== House %d (%s): %s ===", house, "raw" if is_raw else "cleaned", ", ".join(names))
    df = load_refit_csv(csv_path)

    # Find aggregate and appliance columns
    agg_col = find_aggregate_col(df)
    target_col = find_appliance_col(df, house, appliance_map,
                                    target_labels=["washing machine"])
    if target_col is None and not is_raw:
        logger.warning("House %d: no Washing Machine column found, skipping", house)
        return None

    # Identify all power columns
    power_cols = [c for c in df.columns if re.match(r"(aggregate|appliance\s*\d+)", c, re.IGNORECASE)]

    # Timestamps
    df = parse_timestamps(df, log_path=None)
    df = df.dropna(subset=["utc"]).copy()

    # Audit BEFORE sorting/deduplication so ordering and duplicate counts are real
    ch_audit, summ = audit_raw(df, power_cols, agg_col)
    ch_audit.to_csv(metrics_dir / f"house{house}_{pfx}audit_channels.csv", index=False)
    import json
    (metrics_dir / f"house{house}_{pfx}audit_summary.json").write_text(
        json.dumps(summ, indent=2, default=str))

    # Deduplication
    df, dedup_stats = deduplicate(df, power_cols,
                                  log_path=metrics_dir / f"house{house}_{pfx}duplicates_log.csv")
    summ.update(dedup_stats)
    (metrics_dir / f"house{house}_{pfx}audit_summary.json").write_text(
        json.dumps(summ, indent=2, default=str))

    # Invalid values
    df = flag_invalid_values(df, power_cols, agg_col,
                             is_solar_house=is_solar,
                             log_path=metrics_dir / f"house{house}_{pfx}modified_values_log.csv")

    # Resample to 1 minute
    df_1min, coverage_df, _ = resample_to_1min(df, power_cols)
    coverage_df["house"] = house
    coverage_df.to_csv(metrics_dir / f"house{house}_{pfx}coverage.csv", index=False)

    # Provider fill detection (cleaned data only)
    if not is_raw:
        other = [c for c in power_cols if c != agg_col]
        df_1min = flag_suspected_provider_fill(df_1min, agg_col, other_cols=other)
        fill_counts = {
            "house": house,
            "suspected_fill_minutes": int(df_1min.get(f"{agg_col}_suspected_fill", pd.Series(dtype=bool)).sum()),
        }
        pd.DataFrame([fill_counts]).to_csv(
            metrics_dir / f"house{house}_{pfx}suspected_fill.csv", index=False)

    # Issues column handling
    issues_col = None
    candidates = [c for c in df.columns if "issue" in c.lower()]
    if candidates:
        issues_col = candidates[0]
        logger.info("House %d: Issues column found in CSV: %s (README documents it: %s)",
                    house, issues_col, issues_info.get("documented"))
    else:
        logger.info("House %d: no Issues column in CSV (README mention: %s)",
                    house, issues_info.get("documented"))

    # Short-gap interpolation (never on target)
    df_1min = interpolate_short_gaps(df_1min, power_cols=[c for c in df_1min.columns
                                                           if not c.endswith(("_valid_count",
                                                                              "_orig_missing",
                                                                              "_interpolated",
                                                                              "_final_missing",
                                                                              "_suspected_fill"))],
                                     target_col=None if is_raw else target_col)  # Section 1 cleans every channel alike

    # Outages
    base_cols = [c for c in power_cols if c in df_1min.columns]
    outages = detect_outages(df_1min, base_cols)
    if outages.empty:
        outages = pd.DataFrame(columns=["channel", "start_utc", "end_utc", "duration_min"])
    outages["house"] = house
    outages.to_csv(metrics_dir / f"house{house}_{pfx}outages.csv", index=False)
    logger.info("House %d: %d outages (>=3 min)%s", house, len(outages),
                f", longest {int(outages['duration_min'].max())} min" if len(outages) else "")


    # Issues flag -> per-minute boolean (any flagged source reading in the minute)
    if issues_col is not None and issues_col in df.columns:
        iss = pd.to_numeric(df.set_index("utc")[issues_col], errors="coerce")
        iss_1min = iss.resample("1min").max().reindex(df_1min.index)
        df_1min["issue_flag"] = (iss_1min.fillna(0) > 0)
        logger.info("House %d: %d minutes carry Issues=1 (%.2f%%)", house,
                    int(df_1min["issue_flag"].sum()), df_1min["issue_flag"].mean() * 100)
        pd.DataFrame([{"house": house, "issue_minutes": int(df_1min["issue_flag"].sum()),
                       "issue_pct": round(float(df_1min["issue_flag"].mean() * 100), 3)}]
                     ).to_csv(metrics_dir / f"house{house}_{pfx}issue_flags.csv", index=False)

    # Save processed parquet
    suffix = "raw_clean" if is_raw else "clean"
    out_path = out_dir / f"house{house}_{suffix}_1min.parquet"
    df_1min.to_parquet(out_path)
    logger.info("Saved: %s (%d rows)", out_path, len(df_1min))

    # Attach metadata for downstream use (after saving: attrs are serialised into parquet)
    if is_raw:
        df_1min.attrs["raw_df"] = df
    df_1min.attrs["house"] = house
    df_1min.attrs["agg_col"] = agg_col
    df_1min.attrs["target_col"] = target_col
    df_1min.attrs["issues_col"] = issues_col

    return df_1min


### 5.4 Locate the exact REFIT files

The file lookup uses parsed house numbers rather than substring matching, so House 1 is not confused with House 10 or House 11.

In [ ]:
def _house_number(path: Path) -> Optional[int]:
    m = re.search(r"house[_\s-]*(\d+)", path.stem, re.IGNORECASE) or re.search(r"(\d+)", path.stem)
    return int(m.group(1)) if m else None


def find_house_csvs(directory: Path, house: int) -> list:
    """All CSV files of one house, ordered by part number (raw data ships as ..._Part1/_Part2)."""
    found = [p for p in directory.rglob("*.csv") if _house_number(p) == house]

    def part_key(p: Path):
        m = re.search(r"part[_\s-]*(\d+)", p.stem, re.IGNORECASE)
        return (int(m.group(1)) if m else 0, p.name)
    return sorted(found, key=part_key)


def find_house_csv(directory: Path, house: int) -> Optional[Path]:
    """First CSV of a house (House1 != House10/11)."""
    files = find_house_csvs(directory, house)
    return files[0] if files else None


## 6. Washing-machine event detection

The EDA needs a consistent way to turn minute-level power readings into operating events. I use the REFIT-specific settings reported in Table 2 of Manca and Massidda (2022): a candidate ON minute is strictly above 20 W, ON runs separated by fewer than 13 fully observed OFF minutes are merged, and the resulting event is retained when its elapsed duration is at least 27 minutes.

The order is important. I merge short pauses first and apply the 27-minute duration rule to the merged event. If I filtered each short ON run before merging, a washing-machine cycle with intermittent drum activity could be split into several short fragments and discarded. Missing minutes are not treated as OFF and are never bridged.

Events longer than 240 minutes are marked `suspect_fill`. This is my quality-control rule, not a threshold from the cited paper and not proof that the event is invalid. Suspect intervals remain visible in the audit output but are excluded from regression targets and reference activity labels. Cycle energy includes all valid measured power inside the event envelope, including short measured pauses; interpolated values are not counted as measured energy.

### 6.1 Define and detect operating events

The event detector applies one shared rule to every household: candidate ON minutes are strictly above 20 W; valid OFF pauses shorter than 13 minutes are merged; and the merged event is retained only if its elapsed duration is at least 27 minutes. The merge happens before the duration filter so intermittent drum activity is not split into short fragments.

In [ ]:
"""
Single detect_events function used in EDA, Q10 sensitivity, and model evaluation.
Fixed parameters from Manca & Massidda (2022) Table 2 for REFIT washing machine:
  threshold = 20 W, min_on_duration = 27 min, min_off_duration = 13 min.
"""
import logging
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Published REFIT washing-machine parameters
# Source: Manca, M.M. and Massidda, L. (2022), Table 2. DOI: 10.2478/caim-2022-0004
DEFAULT_THRESHOLD_W   = 20.0
MIN_ON_DURATION_MIN   = 27
MIN_OFF_DURATION_MIN  = 13
MAX_CYCLE_DURATION_MIN = 240


@dataclass
class WMEvent:
    house: int
    cycle_id: int
    start_utc: pd.Timestamp
    end_utc: pd.Timestamp
    start_local: pd.Timestamp
    end_local: pd.Timestamp
    elapsed_min: int
    active_min: int         # minutes above threshold
    peak_w: float
    mean_active_w: float
    energy_kwh: float
    missing_minutes_inside: int
    imputed_minutes_inside: int
    issue_minutes_inside: int
    incomplete: bool
    suspect_fill: bool
    threshold_w: float
    threshold_method: str


def detect_events(
    power: pd.Series,
    house: int = 0,
    threshold_w: float = DEFAULT_THRESHOLD_W,
    threshold_method: str = "fixed_20W",
    min_on_min: int = MIN_ON_DURATION_MIN,
    min_off_min: int = MIN_OFF_DURATION_MIN,
    orig_missing: Optional[pd.Series] = None,
    imputed: Optional[pd.Series] = None,
    issue_flag: Optional[pd.Series] = None,
) -> list[WMEvent]:
    """
    Detect washing-machine operating events.

    Algorithm (order is important — merge first, then filter by duration):
    Step 1. Candidate ON minutes: valid power strictly > threshold_w.
            Missing minutes are neither ON nor OFF.
    Step 2. Find runs of consecutive candidate ON minutes.
    Step 3. Merge two ON runs when the gap between them is < min_off_min AND
            every minute in the gap is valid (not missing).
            A gap with any missing minute is NOT merged.
    Step 4. Keep events whose elapsed duration >= min_on_min.
    Step 5. Flag incomplete: event touches a missing minute or the series boundary.
    Step 6. Flag suspect_fill when elapsed duration exceeds MAX_CYCLE_DURATION_MIN.
            This is a conservative project quality-control heuristic, not proof of
            provider corruption and not a published REFIT activation parameter.

    Returns a list of WMEvent dataclasses.
    """
    if not isinstance(power.index, pd.DatetimeIndex):
        raise ValueError("power must have a DatetimeIndex")

    power = power.copy()
    # Work on a complete 1-minute grid. Rows that were dropped upstream (e.g. missing
    # targets) must appear as NaN so events are never merged or measured across them.
    if len(power) > 1:
        full = pd.date_range(power.index[0], power.index[-1], freq="1min", tz=power.index.tz)
        if len(full) != len(power):
            power = power.reindex(full)
        if imputed is not None:
            imputed = imputed.reindex(power.index)
        if issue_flag is not None:
            issue_flag = issue_flag.reindex(power.index)
    local = power.index.tz_convert("Europe/London")

    # Boolean masks aligned to power index
    is_missing  = power.isna()
    is_on       = (~is_missing) & (power > threshold_w)
    is_imputed  = imputed.reindex(power.index).fillna(False).astype(bool) if imputed is not None else pd.Series(False, index=power.index)
    has_issue   = issue_flag.reindex(power.index).fillna(False).astype(bool) if issue_flag is not None else pd.Series(False, index=power.index)

    if is_on.sum() == 0:
        logger.info("House %d: no ON minutes found above %.1f W", house, threshold_w)
        return []

    # --- Step 2: Find ON runs ---
    on_run_id = (is_on != is_on.shift()).cumsum()
    on_runs = []
    for rid, grp in is_on.groupby(on_run_id):
        if grp.iloc[0]:  # This group is ON
            on_runs.append((grp.index[0], grp.index[-1]))

    if not on_runs:
        return []

    # --- Step 3: Merge runs separated by short valid gaps ---
    merged = [list(on_runs[0])]
    for start, end in on_runs[1:]:
        gap_start = merged[-1][1] + pd.Timedelta(minutes=1)
        gap_end = start - pd.Timedelta(minutes=1)
        if gap_start > gap_end:
            # Runs are adjacent
            merged[-1][1] = end
            continue
        gap_slice = is_missing[gap_start:gap_end]
        gap_len = len(gap_slice)
        gap_has_missing = gap_slice.any()

        if gap_len < min_off_min and not gap_has_missing:
            merged[-1][1] = end   # Merge
        else:
            merged.append([start, end])

    # --- Steps 4-6: Filter and annotate ---
    events = []
    for cycle_id, (ev_start, ev_end) in enumerate(merged):
        ev_slice = power[ev_start:ev_end]
        elapsed = len(ev_slice)

        if elapsed < min_on_min:
            continue  # Too short — discard

        active_mask = (~is_missing[ev_start:ev_end]) & (power[ev_start:ev_end] > threshold_w)
        active_min = int(active_mask.sum())
        missing_inside = int(is_missing[ev_start:ev_end].sum())
        imputed_inside = int(is_imputed[ev_start:ev_end].sum())
        issue_inside = int(has_issue[ev_start:ev_end].sum())

        # Peak and mean describe active (>threshold) observations; cycle energy includes
        # ALL valid measured power inside the event envelope, including short OFF pauses.
        active_power = ev_slice[active_mask]
        measured_mask = (~is_missing[ev_start:ev_end]) & (~is_imputed[ev_start:ev_end])
        measured_power = ev_slice[measured_mask]
        peak_w = float(active_power.max()) if len(active_power) else np.nan
        mean_active_w = float(active_power.mean()) if len(active_power) else np.nan
        energy_kwh = float(measured_power.sum() / 60_000.0) if len(measured_power) else 0.0

        # Incomplete events touch the series boundary, contain missing minutes, or sit
        # immediately beside a missing run. This catches cycles split by an outage.
        start_pos = power.index.get_loc(ev_start)
        end_pos = power.index.get_loc(ev_end)
        prev_missing = bool(is_missing.iloc[start_pos - 1]) if start_pos > 0 else False
        next_missing = bool(is_missing.iloc[end_pos + 1]) if end_pos + 1 < len(power) else False
        touches_boundary = (start_pos == 0) or (end_pos == len(power) - 1)
        incomplete = bool(touches_boundary or missing_inside > 0 or prev_missing or next_missing)

        # Long events are flagged as suspected provider fill / stuck-channel events.
        # Flat heating plateaus are retained because they can be genuine washing phases.
        suspect_fill = bool(elapsed > MAX_CYCLE_DURATION_MIN)

        events.append(WMEvent(
            house=house,
            cycle_id=cycle_id,
            start_utc=ev_start,
            end_utc=ev_end,
            start_local=local[power.index.get_loc(ev_start)] if not isinstance(power.index.get_loc(ev_start), slice) else local[power.index.get_loc(ev_start)],
            end_local=local[power.index.get_loc(ev_end)] if not isinstance(power.index.get_loc(ev_end), slice) else local[power.index.get_loc(ev_end)],
            elapsed_min=elapsed,
            active_min=active_min,
            peak_w=peak_w,
            mean_active_w=mean_active_w,
            energy_kwh=energy_kwh,
            missing_minutes_inside=missing_inside,
            imputed_minutes_inside=imputed_inside,
            issue_minutes_inside=issue_inside,
            incomplete=incomplete,
            suspect_fill=suspect_fill,
            threshold_w=threshold_w,
            threshold_method=threshold_method,
        ))

    logger.info("House %d [%.0fW]: %d events detected (%d complete, %d suspect_fill)",
                house, threshold_w,
                len(events),
                sum(1 for e in events if not e.incomplete and not e.suspect_fill),
                sum(1 for e in events if e.suspect_fill))
    return events


### 6.2 Convert events into tables and state masks

These helpers convert the detected events into a cycle table, a minute-level event-envelope series and masks for selecting event intervals. The same event definition is reused later in the model evaluation so the meaning of ON is consistent.

In [ ]:
def events_to_df(events: list[WMEvent]) -> pd.DataFrame:
    if not events:
        return pd.DataFrame()
    return pd.DataFrame([e.__dict__ for e in events])


def events_to_binary_series(events: list[WMEvent], index: pd.DatetimeIndex,
                            include_suspect: bool = False) -> pd.Series:
    """Convert event envelopes to an aligned ON/OFF series.

    Ground-truth labels exclude suspect-fill events by default. Predicted events
    may be included even when longer than the quality-control duration threshold,
    because overlong model activations should be penalised rather than hidden.
    Incomplete observed events are retained for per-minute scoring on their valid
    portions; cycle-level summaries separately exclude incomplete events.
    """
    s = pd.Series(False, index=index, dtype=bool)
    for ev in events:
        if ev.suspect_fill and not include_suspect:
            continue
        s.loc[ev.start_utc:ev.end_utc] = True
    return s


def event_interval_mask(events: list[WMEvent], index: pd.DatetimeIndex,
                        suspect_only: bool = True) -> pd.Series:
    """Return a mask for the full interval of suspect events on a complete grid."""
    mask = pd.Series(False, index=index, dtype=bool)
    for ev in events:
        if suspect_only and not ev.suspect_fill:
            continue
        mask.loc[ev.start_utc:ev.end_utc] = True
    return mask


### 6.3 Bridge short label gaps and calculate the Q10 sensitivity threshold

Short gaps can be bridged for event labelling only when the gap is internal and not protected by an aggregate-based provider-fill flag. The measured regression target remains missing at those timestamps. Q10 is calculated as a secondary sensitivity check; it does not replace the fixed 20 W primary rule.

In [ ]:
def label_events(target: pd.Series, house: int = 0,
                 issue_flag: Optional[pd.Series] = None,
                 protected_mask: Optional[pd.Series] = None):
    """Create event labels without modifying measured target values.

    Internal 1–2 minute target gaps may be bridged for event continuity only.
    A gap overlapping an aggregate-based suspected-provider-fill mask is protected
    and must remain missing; otherwise the labelling-only interpolation could undo
    the provider-fill exclusion. Issues minutes are *not* protected here because
    they are retained to avoid fragmenting an event; they are excluded from model
    training/scoring separately.

    Returns (events, bridged_series, imputed_mask).
    """
    tmp_frame = target.to_frame("t")
    if protected_mask is not None:
        tmp_frame["aggregate_suspected_fill"] = (
            protected_mask.reindex(target.index).fillna(False).astype(bool)
        )
    tmp = interpolate_short_gaps(tmp_frame, ["t"], target_col=None)
    bridged, imputed = tmp["t"], tmp["t_interpolated"]
    events = detect_events(bridged, house=house, imputed=imputed, issue_flag=issue_flag)
    return events, bridged, imputed


def compute_q10_threshold(power: pd.Series, min_readings: int = 100) -> Optional[float]:
    """
    Compute Q10 (10th percentile) of valid readings strictly > 5 W.
    Returns None if fewer than min_readings candidates or fewer than 5 unique values.
    """
    candidates = power[power > 5.0].dropna()
    if len(candidates) < min_readings or candidates.nunique() < 5:
        logger.info("Q10: insufficient data (%d readings, %d unique values)",
                    len(candidates), candidates.nunique())
        return None
    q10 = float(candidates.quantile(0.10))
    if q10 < 10.0:
        logger.warning("Q10=%.2f W is below 10 W (near noise floor) — result labelled as such", q10)
    return q10


## 7. Aggregate-only feature engineering

The predictive input must reflect what a whole-house smart meter can provide. I therefore use aggregate power only, not refrigerator, washing-machine or other appliance sub-meter readings.

The lag features describe recent changes, while rolling mean, maximum and standard deviation summarise the recent load level and variability. I include windows up to 60 minutes because a washing-machine cycle is longer than a few minutes and may have distinct phases. These features are calculated separately for each household so a rolling window cannot cross from one home into another.

The HistGradientBoosting features use current and past aggregate values. That is a deliberate causal design choice: it could be used with a live meter feed. Seq2Point uses a centred window instead, which is useful for retrospective disaggregation but requires later aggregate readings.

In [ ]:
"""
Causal aggregate feature engineering.
All features use only current and past values so the pipeline could
run on a live meter feed. This is a design choice, not leakage prevention.
"""
import logging
import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

LAGS_MIN        = [1, 5, 10, 30, 60]
ROLLING_WINDOWS = [5, 15, 30, 60]
MIN_PERIODS_FRAC = 0.8  # require 80% of window to have valid data


def build_features(df_1min: pd.DataFrame, agg_col: str,
                   target_col: str) -> pd.DataFrame:
    """
    Build causal feature matrix for one household.

    - Features: aggregate lags, rolling stats, calendar.
    - Target: washing-machine power (W).
    - Rows with missing aggregate_current OR missing target are dropped.
    - Rows with missing lag/rolling features are kept (HistGBR handles NaN).
    - Never uses any appliance column or household ID.
    """
    feat = pd.DataFrame(index=df_1min.index)

    # Current aggregate
    feat["aggregate_current"] = df_1min[agg_col]

    # Lags
    for lag in LAGS_MIN:
        feat[f"aggregate_lag_{lag}"] = df_1min[agg_col].shift(lag)

    # Deltas
    feat["aggregate_delta_1"]  = df_1min[agg_col] - df_1min[agg_col].shift(1)
    feat["aggregate_delta_5"]  = df_1min[agg_col] - df_1min[agg_col].shift(5)

    # Rolling statistics (closed on current minute, causal)
    for w in ROLLING_WINDOWS:
        min_p = max(1, int(np.ceil(MIN_PERIODS_FRAC * w)))
        rolling = df_1min[agg_col].rolling(window=w, min_periods=min_p)
        feat[f"aggregate_roll_mean_{w}"] = rolling.mean()
        feat[f"aggregate_roll_std_{w}"]  = rolling.std()
        feat[f"aggregate_roll_max_{w}"]  = rolling.max()

    # Calendar features (Europe/London local time)
    local = df_1min.index.tz_convert("Europe/London")
    feat["hour"]        = local.hour
    feat["day_of_week"] = local.dayofweek  # 0=Monday

    # Sine/cosine encoding for periodicity
    feat["hour_sin"]    = np.sin(2 * np.pi * local.hour / 24)
    feat["hour_cos"]    = np.cos(2 * np.pi * local.hour / 24)
    feat["dow_sin"]     = np.sin(2 * np.pi * local.dayofweek / 7)
    feat["dow_cos"]     = np.cos(2 * np.pi * local.dayofweek / 7)

    # Target
    feat["target_w"] = df_1min[target_col]

    # Keep every minute with a valid aggregate: the model can predict there, and the ground-truth activity
    # labels (true_on) must stay continuous. Rows whose target is missing or excluded keep target_w = NaN;
    # training and scoring use only rows with a valid target_w.
    before = len(feat)
    feat = feat.dropna(subset=["aggregate_current"])
    logger.info("Features: %d rows with a valid aggregate (%d dropped); %d also have a valid target",
                len(feat), before - len(feat), int(feat["target_w"].notna().sum()))

    # Safety check: no appliance channel in features
    _assert_no_appliance_leak(feat, target_col)

    return feat


def _assert_no_appliance_leak(feat: pd.DataFrame, target_col: str) -> None:
    import re
    bad = [c for c in feat.columns
           if re.match(r"appliance\s*\d+", c, re.IGNORECASE)
           and c != "target_w"]
    if bad:
        raise AssertionError(f"Appliance columns found in feature matrix: {bad}")
    if "house" in feat.columns or "household" in feat.columns:
        raise AssertionError("Household ID found in feature matrix — remove it")


NON_FEATURE_COLS = {"target_w", "target_raw", "true_on"}   # labels / bookkeeping, never model inputs


def get_feature_cols(feat: pd.DataFrame) -> list[str]:
    return [c for c in feat.columns if c not in NON_FEATURE_COLS]

## 8. Baselines and HistGradientBoosting

I include two baselines before evaluating a machine-learning model.

- **Always-off:** predicts 0 W at every minute. It is deliberately simple, but it is an important reference because the target is zero for most of the monitoring period.
- **Hour-of-week profile:** predicts the training-house average washing-machine power for the same weekday and hour. This checks whether a simple usage schedule explains some of the signal without learning a detailed aggregate-to-appliance relationship.

The primary tabular model is `HistGradientBoostingRegressor`. I chose it because it handles nonlinear feature interactions and missing lag/rolling features, while remaining much lighter to train than a large ensemble of deep trees on millions of rows. A deterministic per-house training cap controls memory and runtime. The cap is applied to training homes only; each held-out home is evaluated on its full valid test support.

In [ ]:
"""
Baseline models:
  1. AlwaysOffBaseline  - predict 0 W at every minute.
  2. HourOfWeekProfile  - mean target per (weekday, hour) from training houses.
"""
import numpy as np
import pandas as pd


class AlwaysOffBaseline:
    def predict(self, feat: pd.DataFrame) -> np.ndarray:
        return np.zeros(len(feat))


class HourOfWeekProfile:
    """
    Training-derived baseline: mean WM power per (day_of_week, hour) bin.
    Uses only training labels — never held-out target.
    """
    def __init__(self):
        self.profile_: dict[tuple, float] = {}
        self.global_mean_: float = 0.0

    def fit(self, feat: pd.DataFrame) -> "HourOfWeekProfile":
        df = feat[["day_of_week", "hour", "target_w"]].dropna()
        self.global_mean_ = float(df["target_w"].mean())
        profile = df.groupby(["day_of_week", "hour"])["target_w"].mean()
        self.profile_ = profile.to_dict()
        return self

    def predict(self, feat: pd.DataFrame) -> np.ndarray:
        keys = pd.MultiIndex.from_arrays([feat["day_of_week"].astype(int),
                                          feat["hour"].astype(int)])
        prof = pd.Series(self.profile_)
        prof.index = pd.MultiIndex.from_tuples(prof.index)
        return prof.reindex(keys).fillna(self.global_mean_).to_numpy()

"""
Primary model: HistGradientBoostingRegressor.
Fixed config, deterministic training-row cap per household per fold.
"""
import logging, time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

logger = logging.getLogger(__name__)

MAX_TRAIN_ROWS_PER_HOUSE = 200_000
RANDOM_SEED = 42

HGBR_CONFIG = dict(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=RANDOM_SEED,
)


def sample_house(feat: pd.DataFrame, max_rows: int = MAX_TRAIN_ROWS_PER_HOUSE,
                 seed: int = RANDOM_SEED) -> pd.DataFrame:
    """
    Deterministic proportional sample of one house's feature rows.
    Preserves natural ON/OFF ratio.
    """
    if len(feat) <= max_rows:
        return feat
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(feat), size=max_rows, replace=False)
    sampled = feat.iloc[sorted(idx)]

    orig_on = (feat["target_w"] > 20).mean()
    samp_on = (sampled["target_w"] > 20).mean()
    logger.info("Sampled %d/%d rows | ON proportion: orig=%.3f sampled=%.3f",
                max_rows, len(feat), orig_on, samp_on)
    return sampled


def fit_model(train_feat: pd.DataFrame) -> tuple[HistGradientBoostingRegressor, float]:
    X = train_feat[get_feature_cols(train_feat)]
    y = train_feat["target_w"].values
    model = HistGradientBoostingRegressor(**HGBR_CONFIG)
    t0 = time.time()
    model.fit(X, y)
    elapsed = time.time() - t0
    logger.info("HistGBR fit: %.1f s", elapsed)
    return model, elapsed


def predict(model: HistGradientBoostingRegressor,
            feat: pd.DataFrame) -> np.ndarray:
    X = feat[get_feature_cols(feat)]
    t0 = time.time()
    preds = model.predict(X)
    elapsed = time.time() - t0
    preds = np.clip(preds, 0, None)  # clip negative predictions to 0
    # Inference diagnostics must not read target labels (especially on LOHO test houses).
    # Report the prediction distribution only; calibration/error checks belong in
    # metric code after predictions have been aligned with valid held-out targets.
    p99_pred = float(np.percentile(preds, 99)) if len(preds) else float("nan")
    logger.info("HistGBR predict: %.1f s | prediction p99=%.1f W", elapsed, p99_pred)
    return preds

## 9. Seq2Point CNN

I include Seq2Point because appliance operation is sequential: a minute's washing-machine power is related to what the aggregate signal was doing before and after that minute. The network receives 121 consecutive aggregate readings (60 minutes on either side plus the centre minute) and predicts one washing-machine power value at the centre.

The architecture is intentionally compact: three 1-D convolution blocks extract local temporal patterns, followed by a small fully connected regression head. The target is continuous power in watts. A weighted Huber loss gives additional weight to active centre minutes while retaining the natural ON/OFF distribution in the training sample. This is meant to reduce the tendency to minimise loss by predicting near-zero power everywhere; it does not change the test distribution.

The input scaler is fitted on training windows only. Early stopping uses a time-separated validation tail drawn from the five training households, not the held-out home. The centred window uses future aggregate context, so the model is appropriate for offline analysis. I do not present it as a real-time causal model.

### 9.1 CNN definition and sequence-window construction

The model uses 1-D convolutions because the input is a one-channel time series. The sequence builder accepts only windows with complete aggregate context; training windows additionally need a valid centre target. This keeps the held-out household usable for prediction without requiring its target values.

In [ ]:
class Seq2PointCNN(nn.Module):
    """Compact 1-D CNN: 121 aggregate readings -> centre-minute appliance power."""
    def __init__(self, window_size=SEQ_WINDOW_MINUTES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(8),
            nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(64 * 8, 128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)


def sequence_candidates(df_1min, agg_col, target_col=None, window=SEQ_WINDOW_MINUTES,
                        require_valid_target=True):
    """Find complete aggregate windows; training additionally requires a centre target.

    The input DataFrame must retain the complete one-minute index. Any aggregate
    gap invalidates every window crossing it; no missing time is bridged.
    """
    if not isinstance(df_1min.index, pd.DatetimeIndex):
        raise ValueError('Expected a DatetimeIndex.')
    if len(df_1min) < window:
        return None, np.array([], dtype=np.int64), np.array([], dtype=np.int64)
    agg = pd.to_numeric(df_1min[agg_col], errors='coerce').to_numpy(dtype=np.float32)
    y = None
    if require_valid_target:
        if target_col is None:
            raise ValueError('target_col is required when require_valid_target=True.')
        y = pd.to_numeric(df_1min[target_col], errors='coerce').to_numpy(dtype=np.float32)
    windows = sliding_window_view(agg, window_shape=window)
    half = window // 2
    centres = np.arange(half, half + len(windows), dtype=np.int64)
    kept = []
    for start in range(0, len(windows), 100_000):
        end = min(start + 100_000, len(windows))
        valid = np.isfinite(windows[start:end]).all(axis=1)
        if require_valid_target:
            valid &= np.isfinite(y[centres[start:end]])
        local = np.flatnonzero(valid)
        if len(local):
            kept.append(local + start)
    positions = np.concatenate(kept) if kept else np.array([], dtype=np.int64)
    return windows, positions, centres[positions]


def sample_sequence_windows(df_1min, agg_col, target_col, house,
                            max_windows=SEQ_MAX_WINDOWS_PER_HOUSE, seed_offset=0):
    windows, positions, centres = sequence_candidates(df_1min, agg_col, target_col)
    if len(positions) == 0:
        return np.empty((0, SEQ_WINDOW_MINUTES), dtype=np.float32), np.empty(0, dtype=np.float32), 0, 0
    y = pd.to_numeric(df_1min[target_col], errors='coerce').to_numpy(dtype=np.float32)
    if len(positions) > max_windows:
        rng = np.random.default_rng(RANDOM_SEED + int(house) + int(seed_offset))
        selected = np.sort(rng.choice(len(positions), size=max_windows, replace=False))
        positions = positions[selected]
        centres = centres[selected]
    X = np.asarray(windows[positions], dtype=np.float32)
    target = y[centres]
    n_on = int((target > DEFAULT_THRESHOLD_W).sum())
    return X, target, len(positions), n_on


### 9.2 Train Seq2Point using training households only

The training routine fits input scaling on training windows, reserves a later validation segment from the training households, and selects the checkpoint using validation loss. The weighted Huber loss gives more influence to active centre targets without changing the natural test distribution. The held-out house is not used for early stopping.

In [ ]:
def _weighted_huber(pred, target, weights):
    per_item = F.smooth_l1_loss(pred, target, beta=0.05, reduction='none')
    return (per_item * weights).mean()


def train_seq2point(train_house_frames, feature_meta, device=None):
    """Train Seq2Point using only training houses and an internal chronological validation tail.

    Each training household is split by time before windows are created. Train and
    validation windows are built separately, so no 121-minute window can overlap
    the split boundary. The held-out LOHO household is never passed to this function.
    Normalisation is fitted on training windows only; early stopping uses validation
    windows from training households only.
    """
    from copy import deepcopy
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    train_X, train_y, val_X, val_y, sampling_rows = [], [], [], [], []

    for house, dfh in train_house_frames.items():
        agg_col, target_col = feature_meta[house]
        split_pos = int(len(dfh) * (1.0 - SEQ_VAL_FRACTION))
        if split_pos <= SEQ_WINDOW_MINUTES or (len(dfh) - split_pos) <= SEQ_WINDOW_MINUTES:
            logger.warning('House %s has insufficient time span for a disjoint Seq2Point validation split.', house)
            continue

        # Separate segments first. Each window is wholly contained in one segment.
        train_seg = dfh.iloc[:split_pos]
        val_seg = dfh.iloc[split_pos:]
        Xtr, ytr, ntr, ontr = sample_sequence_windows(
            train_seg, agg_col, target_col, house,
            max_windows=SEQ_MAX_WINDOWS_PER_HOUSE, seed_offset=0)
        Xva, yva, nva, onva = sample_sequence_windows(
            val_seg, agg_col, target_col, house,
            max_windows=SEQ_VAL_MAX_WINDOWS_PER_HOUSE, seed_offset=100_000)

        if ntr == 0:
            logger.warning('House %s contributes no Seq2Point training windows.', house)
            continue
        if nva == 0:
            logger.warning('House %s contributes no Seq2Point validation windows.', house)
            continue

        train_X.append(Xtr); train_y.append(ytr)
        val_X.append(Xva); val_y.append(yva)
        sampling_rows.append({
            'house': int(house), 'train_windows': int(ntr), 'train_on_centres': int(ontr),
            'train_on_pct': round(ontr / max(ntr, 1) * 100, 3),
            'validation_windows': int(nva), 'validation_on_centres': int(onva),
            'validation_on_pct': round(onva / max(nva, 1) * 100, 3),
            'split_time_utc': str(dfh.index[split_pos]),
        })

    if not train_X or not val_X:
        raise RuntimeError('Seq2Point requires valid, disjoint train and validation windows from training households.')

    Xtr = np.concatenate(train_X, axis=0)
    ytr = np.concatenate(train_y, axis=0)
    Xva = np.concatenate(val_X, axis=0)
    yva = np.concatenate(val_y, axis=0)

    # Fit input scaling on training windows only.
    x_mean = float(Xtr.mean())
    x_std = float(Xtr.std())
    if not np.isfinite(x_std) or x_std < 1e-6:
        x_std = 1.0
    Xtr = ((Xtr - x_mean) / x_std).astype(np.float32)
    Xva = ((Xva - x_mean) / x_std).astype(np.float32)
    ytr_s = (ytr / SEQ_TARGET_SCALE_W).astype(np.float32)
    yva_s = (yva / SEQ_TARGET_SCALE_W).astype(np.float32)

    wtr = np.where(ytr > DEFAULT_THRESHOLD_W, SEQ_ACTIVE_LOSS_WEIGHT, 1.0).astype(np.float32)
    wva = np.where(yva > DEFAULT_THRESHOLD_W, SEQ_ACTIVE_LOSS_WEIGHT, 1.0).astype(np.float32)

    train_ds = TensorDataset(torch.from_numpy(Xtr[:, None, :]), torch.from_numpy(ytr_s), torch.from_numpy(wtr))
    val_ds = TensorDataset(torch.from_numpy(Xva[:, None, :]), torch.from_numpy(yva_s), torch.from_numpy(wva))
    generator = torch.Generator()
    generator.manual_seed(RANDOM_SEED)
    train_loader = DataLoader(train_ds, batch_size=SEQ_BATCH_SIZE, shuffle=True, num_workers=0,
                              pin_memory=(device == 'cuda'), generator=generator)
    val_loader = DataLoader(val_ds, batch_size=SEQ_BATCH_SIZE, shuffle=False, num_workers=0,
                            pin_memory=(device == 'cuda'))

    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    model = Seq2PointCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    best_val = float('inf')
    best_epoch = 0
    best_state = None
    no_improve = 0
    history = []
    t0 = time.time()

    for epoch in range(SEQ_EPOCHS):
        model.train()
        train_loss_sum = 0.0
        train_seen = 0
        for xb, yb, wb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            wb = wb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = _weighted_huber(pred, yb, wb)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach()) * len(xb)
            train_seen += len(xb)
        train_loss = train_loss_sum / max(train_seen, 1)

        model.eval()
        val_loss_sum = 0.0
        val_seen = 0
        with torch.no_grad():
            for xb, yb, wb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                wb = wb.to(device, non_blocking=True)
                pred = model(xb)
                loss = _weighted_huber(pred, yb, wb)
                val_loss_sum += float(loss) * len(xb)
                val_seen += len(xb)
        val_loss = val_loss_sum / max(val_seen, 1)
        history.append({'epoch': epoch + 1, 'train_weighted_huber': train_loss,
                        'validation_weighted_huber': val_loss})
        print(f'Seq2Point epoch {epoch+1:02d}/{SEQ_EPOCHS}: train={train_loss:.5f} val={val_loss:.5f}')

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_epoch = epoch + 1
            best_state = deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= SEQ_PATIENCE:
                print(f'Early stopping at epoch {epoch+1}; best epoch={best_epoch}.')
                break

    if best_state is None:
        raise RuntimeError('Seq2Point did not produce a valid validation checkpoint.')
    model.load_state_dict(best_state)
    fit_seconds = time.time() - t0
    sampling = pd.DataFrame(sampling_rows)
    history_df = pd.DataFrame(history)
    print(f'Seq2Point best checkpoint: epoch {best_epoch}, validation loss={best_val:.5f}; '
          f'train windows={len(Xtr):,}, validation windows={len(Xva):,}')
    return model, {
        'x_mean': x_mean, 'x_std': x_std, 'device': device,
        'fit_time_s': fit_seconds, 'best_epoch': best_epoch, 'best_val_loss': best_val,
        'n_train_windows': len(Xtr), 'n_validation_windows': len(Xva),
        'n_train_on_centres': int((ytr > DEFAULT_THRESHOLD_W).sum()),
        'sampling': sampling, 'history': history_df,
    }


### 9.3 Predict on an unseen household

Inference uses only the aggregate sequence. The target column is optional and is used only to align/score predictions after inference; it does not enter the CNN or determine which aggregate windows can be predicted.

In [ ]:
@torch.no_grad()
def predict_seq2point(model, scaler, df_test, agg_col, target_col=None, batch_size=2048):
    """Predict wherever aggregate-only 121-minute input context is complete.

    `target_col` is intentionally unused: target availability must not control
    whether an inference prediction is generated.
    """
    windows, positions, centres = sequence_candidates(
        df_test, agg_col, target_col=None, require_valid_target=False)
    if len(positions) == 0:
        return pd.Series(dtype=float), 0.0
    device = scaler['device']
    model.eval()
    chunks = []
    t0 = time.time()
    for start in range(0, len(positions), batch_size):
        pos = positions[start:start + batch_size]
        xb = np.asarray(windows[pos], dtype=np.float32)
        xb = (xb - scaler['x_mean']) / scaler['x_std']
        xt = torch.from_numpy(xb[:, None, :]).to(device)
        pred = model(xt).detach().cpu().numpy() * SEQ_TARGET_SCALE_W
        chunks.append(pred)
    pred = np.clip(np.concatenate(chunks), 0, None)
    elapsed = time.time() - t0
    idx = df_test.index[centres]
    return pd.Series(pred, index=idx, name='y_pred_seq2point'), elapsed

print('Seq2Point architecture, train-only validation and inference utilities defined.')


## 10. Plotting functions

These functions keep the figures consistent across households: common labels, units, time zones and deterministic date selection. The raw-data plots use a short zoom window because plotting two years of 8-second observations in one panel would hide the individual data-quality problems. The full monitoring period is summarised with daily coverage instead.

For EDA and model evaluation, the selected day is based on coverage and cycle statistics rather than a manually chosen example. This makes the plots easier to reproduce and reduces the chance of selecting only a visually attractive result.

### 10.1 Data-quality plots and raw-data zooms

The plotting helpers show daily coverage across the full monitoring period and a two-day window chosen from the recorded data-quality flags. This gives both a long-range view of missingness and a readable view of individual readings before and after cleaning.

In [ ]:
"""
All required figures for the assignment.
Deterministic plot selection — no manual choices.
"""
import logging
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

logger = logging.getLogger(__name__)

FIGSIZE_WIDE = (14, 4)
FIGSIZE_SQUARE = (10, 6)


def _save(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved figure: %s", path)


# ---------------------------------------------------------------------------
# Section 1: Data quality plots
# ---------------------------------------------------------------------------

def _nm(col: str, labels: Optional[dict]) -> str:
    return f"{col} ({labels[col]})" if labels and col in labels else col


def plot_daily_coverage(df_1min: pd.DataFrame, power_cols: list[str],
                        house: int, out_dir: Path,
                        df_raw: Optional[pd.DataFrame] = None,
                        raw_interval_s: float = 8.0, labels: Optional[dict] = None):
    """
    Full-period daily coverage, raw vs cleaned.
    raw     = readings received per day / readings expected at the median interval
    cleaned = % of 1-minute bins that are valid (50% coverage rule) after short-gap fill
    """
    fig, axes = plt.subplots(len(power_cols), 1,
                             figsize=(14, 3 * len(power_cols)), sharex=True)
    if len(power_cols) == 1:
        axes = [axes]
    expected_per_day = 86400.0 / raw_interval_s
    for ax, col in zip(axes, power_cols):
        if col not in df_1min.columns:
            continue
        loc = df_1min.index.tz_convert("Europe/London").tz_localize(None).normalize()
        clean = (~df_1min[col].isna()).groupby(loc).mean() * 100
        ax.plot(clean.index, clean.values, lw=1.2, color="#2c7bb6", label="Cleaned 1-min (valid %)")
        if df_raw is not None and col in df_raw.columns:
            r = df_raw.dropna(subset=[col])
            rloc = r["utc"].dt.tz_convert("Europe/London").dt.tz_localize(None).dt.normalize()
            raw = (rloc.value_counts() / expected_per_day * 100).clip(upper=100)
            raw = raw.reindex(clean.index, fill_value=0).sort_index()   # absent days = 0%, not a diagonal line
            ax.plot(raw.index, raw.values, lw=0.9, color="grey", alpha=0.8,
                    label="Raw 8-s (received % of expected)")
        ax.set_ylim(0, 105)
        ax.set_ylabel("% valid")
        ax.set_title(f"House {house} — {_nm(col, labels)}: daily coverage")
        ax.legend(fontsize=8, loc="lower left")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    fig.autofmt_xdate()
    _save(fig, out_dir / f"house{house}_daily_coverage.png")


def select_zoom_window(df_clean: pd.DataFrame, extra_flag_times=None) -> tuple[pd.Timestamp, str]:
    """
    Deterministic 48 h zoom rule: the two-day window (starting at a UTC midnight)
    with the highest count of data-quality flags = missing minutes (any channel)
    + interpolated minutes + minutes containing a logged invalid value or duplicate.
    Falls back to the earliest two-day window with adequate data if nothing is flagged.
    """
    flag_cols = [c for c in df_clean.columns if c.endswith(("_orig_missing", "_interpolated"))]
    score = pd.Series(0, index=df_clean.index, dtype=float)
    for c in flag_cols:
        score += df_clean[c].fillna(False).astype(float)
    if extra_flag_times is not None and len(extra_flag_times):
        et = pd.DatetimeIndex(extra_flag_times).floor("min")
        et = et[et.isin(score.index)]
        score.loc[et] += 1
    daily = score.groupby(score.index.floor("D")).sum()
    two_day = daily + daily.shift(-1).fillna(0)
    # require the window to hold some real data
    valid_day = (~df_clean.filter(regex=r"^(Aggregate|aggregate)$").iloc[:, 0].isna()
                 ).groupby(df_clean.index.floor("D")).mean() if len(df_clean.filter(regex=r"^(Aggregate|aggregate)$").columns) else None
    if valid_day is not None:
        two_day = two_day[(valid_day >= 0.5) & (valid_day.shift(-1).fillna(0) >= 0.2)]
    if len(two_day) and two_day.max() > 0:
        return two_day.idxmax(), "highest data-quality-flag count in a 48 h window"
    first = two_day.index[0] if len(two_day) else df_clean.index[0].floor("D")
    return first, "no flagged window found; earliest adequate 48 h window"


def plot_before_after_zoom(df_raw: Optional[pd.DataFrame], df_clean: pd.DataFrame,
                           agg_col: str, appliance_cols: list[str],
                           house: int, out_dir: Path, extra_flag_times=None,
                           labels: Optional[dict] = None) -> str:
    """48 h zoom: raw 8-s readings vs cleaned 1-min. Returns the selection reason."""
    zoom_start, reason = select_zoom_window(df_clean, extra_flag_times)
    zoom_end = zoom_start + pd.Timedelta(days=2)
    all_cols = [agg_col] + appliance_cols
    fig, axes = plt.subplots(len(all_cols), 1,
                             figsize=(14, 3 * len(all_cols)), sharex=True)
    if len(all_cols) == 1:
        axes = [axes]
    raw_idx = None
    if df_raw is not None:
        raw_idx = df_raw.set_index("utc").sort_index()
    for ax, col in zip(axes, all_cols):
        if raw_idx is not None and col in raw_idx.columns:
            rs = raw_idx.loc[zoom_start:zoom_end, col].dropna()
            ax.plot(rs.index.tz_convert("Europe/London"), rs.values,
                    lw=0.5, color="grey", alpha=0.6, label="Raw 8 s (before)")
        if col in df_clean.columns:
            cs = df_clean.loc[zoom_start:zoom_end, col]
            li = cs.index.tz_convert("Europe/London")
            ax.plot(li, cs.values, lw=1.2, color="#2c7bb6", label="Cleaned 1 min (after)")
            ic = f"{col}_interpolated"
            if ic in df_clean.columns:
                m = df_clean.loc[zoom_start:zoom_end, ic].fillna(False).astype(bool).values
                if m.any():
                    ax.scatter(li[m], cs.values[m], s=22, color="orange", zorder=5,
                               label="Interpolated (1-2 min gaps)")
            # shade missing spans (outages / unfilled gaps)
            miss = cs.isna().values
            if miss.any():
                ax.fill_between(li, 0, 1, where=miss, transform=ax.get_xaxis_transform(),
                                color="red", alpha=0.12, label="Missing after cleaning")
        ax.set_ylabel("Power (W)")
        ax.set_title(f"House {house} — {_nm(col, labels)}")
        ax.legend(fontsize=7, loc="upper right")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"))
    axes[0].set_xlabel("")
    fig.suptitle(f"Window starts {zoom_start:%Y-%m-%d} UTC — {reason}", fontsize=9, y=1.0)
    fig.autofmt_xdate()
    _save(fig, out_dir / f"house{house}_before_after_zoom.png")
    return f"{zoom_start} ({reason})"


### 10.2 EDA plots

These functions create the representative 24-hour trace, detected-cycle view, household comparison, hourly usage and day-of-week usage. The plot-selection rule is deterministic, so the displayed example can be regenerated from the same data.

In [ ]:
# ---------------------------------------------------------------------------
# Section 2: EDA plots
# ---------------------------------------------------------------------------

def plot_representative_day(df_1min: pd.DataFrame, agg_col: str, target_col: str,
                             events: list[WMEvent], house: int, out_dir: Path):
    """
    Required Plot 1: representative 24-hour aggregate + WM trace.
    Selection: local day containing the complete cycle with energy
    closest to median complete cycle energy, with >=95% joint coverage.
    """
    complete = [e for e in events if not e.incomplete and not e.suspect_fill]
    if not complete:
        logger.warning("House %d: no complete cycles for representative day plot", house)
        return

    energies = [e.energy_kwh for e in complete]
    median_e = float(np.median(energies))
    local = df_1min.index.tz_convert("Europe/London")

    best_event = None
    best_day = None
    min_e_diff = float("inf")

    for ev in complete:
        day = ev.start_local.normalize()
        day_utc_start = day.tz_convert("UTC")
        day_utc_end = day_utc_start + pd.Timedelta(days=1)
        day_slice = df_1min.loc[day_utc_start:day_utc_end]
        if len(day_slice) == 0:
            continue
        agg_cov = day_slice[agg_col].notna().mean()
        tgt_cov = day_slice[target_col].notna().mean()
        if agg_cov >= 0.95 and tgt_cov >= 0.95:
            diff = abs(ev.energy_kwh - median_e)
            if diff < min_e_diff:
                min_e_diff = diff
                best_event = ev
                best_day = day

    if best_day is None:
        # Fallback: highest joint coverage day
        days = pd.Series(local.normalize()).unique()
        best_cov = 0
        for day in days:
            day_utc = day.tz_convert("UTC") if hasattr(day, "tz_convert") else pd.Timestamp(day).tz_localize("UTC")
            sl = df_1min.loc[day_utc:day_utc + pd.Timedelta(days=1)]
            cov = min(sl[agg_col].notna().mean(), sl[target_col].notna().mean())
            if cov > best_cov:
                best_cov = cov
                best_day = day
        logger.warning("House %d: no 95%% coverage day — using highest coverage day (%.0f%%)",
                       house, best_cov * 100)

    day_utc_start = pd.Timestamp(best_day).tz_convert("UTC") if hasattr(best_day, "tz_convert") else pd.Timestamp(best_day).tz_localize("UTC")
    day_utc_end = day_utc_start + pd.Timedelta(days=1)
    sl = df_1min.loc[day_utc_start:day_utc_end]
    local_idx = sl.index.tz_convert("Europe/London")

    fig, axes = plt.subplots(2, 1, figsize=FIGSIZE_WIDE, sharex=True)
    axes[0].plot(local_idx, sl[agg_col], lw=1, color="#333", label="Aggregate")
    axes[0].set_ylabel("Power (W)")
    axes[0].set_title(f"House {house} — {best_day.date()} — Aggregate power")
    axes[0].legend()

    axes[1].plot(local_idx, sl[target_col], lw=1.2, color="#2c7bb6", label="Washing Machine")
    axes[1].axhline(DEFAULT_THRESHOLD_W, color="red", ls="--", lw=0.8, label=f"{DEFAULT_THRESHOLD_W} W threshold")
    if best_event:
        s = best_event.start_local
        e = best_event.end_local
        axes[1].axvspan(s, e, alpha=0.15, color="green", label="Detected cycle")
    axes[1].set_ylabel("Power (W)")
    axes[1].set_title("Washing Machine power")
    axes[1].legend()
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    _save(fig, out_dir / f"house{house}_representative_day.png")


def plot_detected_cycles(df_1min: pd.DataFrame, target_col: str,
                         events: list[WMEvent], house: int, out_dir: Path,
                         n_cycles: int = 2):
    """Required Plot 2: detected washing-machine cycles with event shading."""
    complete = [e for e in events if not e.incomplete and not e.suspect_fill]
    if not complete:
        return
    # Pick cycle with energy closest to median
    energies = sorted([e.energy_kwh for e in complete])
    median_e = energies[len(energies) // 2]
    picked = sorted(complete, key=lambda e: abs(e.energy_kwh - median_e))[:n_cycles]

    for ev in picked:
        pad = pd.Timedelta(minutes=30)
        sl = df_1min.loc[ev.start_utc - pad: ev.end_utc + pad, target_col]
        local_idx = sl.index.tz_convert("Europe/London")

        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(local_idx, sl.values, lw=1.2, color="#2c7bb6", label="WM power")
        ax.axhline(DEFAULT_THRESHOLD_W, color="red", ls="--", lw=0.8, label=f"{DEFAULT_THRESHOLD_W} W")
        ax.axvspan(ev.start_local, ev.end_local, alpha=0.15, color="green",
                   label=f"Event: {ev.elapsed_min} min, {ev.energy_kwh:.3f} kWh")
        ax.set_ylabel("Power (W)")
        ax.set_title(f"House {house} — Cycle {ev.cycle_id} ({ev.start_local.date()})")
        ax.legend(fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        _save(fig, out_dir / f"house{house}_cycle_{ev.cycle_id}.png")


def plot_cross_house_comparison(cycles_df: pd.DataFrame, out_dir: Path):
    """Required Plot 3: energy per cycle and duration by house."""
    complete = cycles_df[~cycles_df["incomplete"] & ~cycles_df["suspect_fill"]]
    if complete.empty:
        return

    houses = sorted(complete["house"].unique())
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Energy boxplot
    data_e = [complete[complete["house"] == h]["energy_kwh"].dropna().values for h in houses]
    axes[0].boxplot(data_e, labels=[f"H{h}" for h in houses], patch_artist=True)
    axes[0].set_ylabel("Energy per cycle (kWh)")
    axes[0].set_title("Energy per cycle by house")

    # Duration boxplot
    data_d = [complete[complete["house"] == h]["elapsed_min"].dropna().values for h in houses]
    axes[1].boxplot(data_d, labels=[f"H{h}" for h in houses], patch_artist=True)
    axes[1].set_ylabel("Duration (min)")
    axes[1].set_title("Cycle duration by house")

    _save(fig, out_dir / "cross_house_comparison.png")


def plot_hour_of_day_usage(cycles_df: pd.DataFrame, out_dir: Path):
    complete = cycles_df[~cycles_df["incomplete"] & ~cycles_df["suspect_fill"]].copy()
    if complete.empty:
        return
    complete["start_hour"] = pd.to_datetime(complete["start_local"]).dt.hour
    fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
    for h in sorted(complete["house"].unique()):
        sub = complete[complete["house"] == h]
        counts = sub["start_hour"].value_counts().sort_index()
        counts = counts / counts.sum()
        ax.plot(counts.index, counts.values, marker="o", label=f"House {h}")
    ax.set_xlabel("Hour of day (local)")
    ax.set_ylabel("Share of cycle starts")
    ax.set_title("Cycle start hour distribution by house")
    ax.legend()
    _save(fig, out_dir / "hour_of_day_usage.png")


def plot_day_of_week_usage(cycles_df: pd.DataFrame, out_dir: Path):
    """Share of complete cycles starting on each weekday (Europe/London), per house."""
    comp = cycles_df[cycles_df["complete"]].copy() if "complete" in cycles_df else \
        cycles_df[~cycles_df["incomplete"] & ~cycles_df["suspect_fill"]].copy()
    if comp.empty:
        return
    dow = pd.to_datetime(comp["start_utc"], utc=True).dt.tz_convert("Europe/London").dt.dayofweek
    comp["dow"] = dow
    fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
    for h in sorted(comp["house"].unique()):
        counts = comp[comp["house"] == h]["dow"].value_counts().reindex(range(7), fill_value=0)
        ax.plot(range(7), counts.values / counts.sum(), marker="o", label=f"House {h}")
    ax.set_xticks(range(7)); ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
    ax.set_ylabel("Share of cycle starts"); ax.set_title("Cycle starts by day of week")
    ax.legend()
    _save(fig, out_dir / "day_of_week_usage.png")


### 10.3 Model and error-analysis plots

These plots use held-out predictions. They show actual versus estimated washing-machine power and help identify where the model misses real events or predicts activity when the washing machine is not operating.

In [ ]:
# ---------------------------------------------------------------------------
# Section 3: Model plots
# ---------------------------------------------------------------------------

def plot_actual_vs_predicted(pred_df: pd.DataFrame, events: list[WMEvent],
                              house: int, loho_results: pd.DataFrame,
                              out_dir: Path):
    """
    Required: actual vs predicted for the median-F1 house.
    Shows the local day containing the median-energy complete actual cycle.
    """
    complete = [e for e in events if not e.incomplete and not e.suspect_fill]
    if not complete:
        logger.warning("House %d: no complete cycles for prediction plot", house)
        return

    energies = [e.energy_kwh for e in complete]
    median_e = float(np.median(energies))

    best_event = min(complete, key=lambda e: abs(e.energy_kwh - median_e))
    day_local = best_event.start_local.normalize()
    day_utc = day_local.tz_convert('UTC')
    day_end = (day_local + pd.DateOffset(days=1)).tz_convert('UTC')
    sl = pred_df.loc[day_utc:day_end]
    if sl.empty:
        sl = pred_df.iloc[:1440]

    local_idx = sl.index.tz_convert("Europe/London")
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    colors = {"y_pred_always_off": "#aaa", "y_pred_profile": "#f4a261", "y_pred_hgbr": "#2c7bb6"}
    labels = {"y_pred_always_off": "Always-off", "y_pred_profile": "Hour-of-week", "y_pred_hgbr": "HistGBR"}

    for ax, col in zip(axes, ["y_pred_always_off", "y_pred_profile", "y_pred_hgbr"]):
        ax.plot(local_idx, sl["y_true"].values, lw=1.5, color="black", label="Actual", alpha=0.9)
        ax.plot(local_idx, sl[col].values, lw=1.2, color=colors[col], label=labels[col], alpha=0.8)
        ax.axhline(DEFAULT_THRESHOLD_W, color="red", ls="--", lw=0.7, alpha=0.5)
        ax.axvspan(best_event.start_local, best_event.end_local, alpha=0.1, color="green")
        ax.set_ylabel("Power (W)")
        ax.legend(fontsize=8)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[0].set_title(f"House {house} — Actual vs Predicted ({day_utc.date()})")
    _save(fig, out_dir / f"house{house}_actual_vs_predicted.png")


def plot_per_house_metrics(loho_results: pd.DataFrame, out_dir: Path):
    """Bar chart: ON-minute MAE and F1 by house and model."""
    models = loho_results["model"].unique()
    houses = sorted(loho_results["house"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(houses))
    w = 0.25

    colors = {"always_off": "#aaa", "hour_of_week_profile": "#f4a261", "HistGBR": "#2c7bb6"}

    for i, model in enumerate(["always_off", "hour_of_week_profile", "HistGBR"]):
        sub = loho_results[loho_results["model"] == model].set_index("house")
        maes = [pd.to_numeric(sub.loc[h, "event_envelope_mae_w"], errors="coerce") if h in sub.index else np.nan for h in houses]
        f1s  = [sub.loc[h, "event_state_f1"] if h in sub.index else np.nan for h in houses]
        label = model.replace("_", " ")
        axes[0].bar(x + i * w, maes, w, label=label, color=colors.get(model, "steelblue"))
        axes[1].bar(x + i * w, f1s,  w, label=label, color=colors.get(model, "steelblue"))

    axes[0].set_xticks(x + w); axes[0].set_xticklabels([f"H{h}" for h in houses])
    axes[0].set_ylabel("Active MAE (W)"); axes[0].set_title("Active-period MAE by house")
    axes[0].legend()

    axes[1].set_xticks(x + w); axes[1].set_xticklabels([f"H{h}" for h in houses])
    axes[1].set_ylabel("Event-state F1"); axes[1].set_title("Event-state F1 by house")
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    _save(fig, out_dir / "per_house_metrics.png")


def plot_failure_case(pred_df: pd.DataFrame, aggregate: pd.Series, start: pd.Timestamp,
                      end: pd.Timestamp, title: str, out_path: Path, pad_min: int = 90):
    """Aggregate + actual + HistGBR prediction around one failure event."""
    a, b = start - pd.Timedelta(minutes=pad_min), end + pd.Timedelta(minutes=pad_min)
    sl = pred_df.loc[a:b]
    ag = aggregate.loc[a:b]
    if sl.empty:
        return
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(ag.index.tz_convert("Europe/London"), ag.values, lw=1, color="#333")
    axes[0].set_ylabel("Aggregate (W)")
    axes[0].set_title(title)
    li = sl.index.tz_convert("Europe/London")
    axes[1].plot(li, sl["y_true"].values, color="black", lw=1.4, label="Actual WM")
    axes[1].plot(li, sl["y_pred_hgbr"].values, color="#2c7bb6", lw=1.2, label="HistGBR")
    axes[1].axhline(DEFAULT_THRESHOLD_W, color="red", ls="--", lw=0.7)
    axes[1].axvspan(start.tz_convert("Europe/London"), end.tz_convert("Europe/London"),
                    alpha=0.12, color="orange", label="Failure event")
    axes[1].set_ylabel("WM power (W)")
    axes[1].legend(fontsize=8)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    _save(fig, out_path)


## 11. LOHO evaluation and metric definitions

The evaluator runs one fold per household. For each fold, five homes provide training data and the remaining home is held out. The test home's aggregate readings are available for inference, but its appliance labels are not used to fit the model, scale inputs, choose the epoch or select thresholds.

Seq2Point can only predict a minute when the full 121-minute aggregate window is available. I therefore calculate the comparison on common support: valid target minutes for which the required aggregate input context is also available for the CNN. I report coverage against both the full one-minute household grid and the household's valid target minutes so the amount of evaluated data remains visible.

The metrics answer different questions:

- **Event-state F1:** agreement after applying the 20 W / 27-minute ON / 13-minute OFF event rule to actual and predicted signals.
- **Per-minute F1:** agreement using the instantaneous 20 W threshold, without event-duration post-processing.
- **Event-envelope MAE:** power error on measured target minutes inside actual event envelopes, including short fully observed pauses merged into a cycle.
- **Energy error:** difference between actual and predicted kWh on the same aligned, valid timestamps.
- **Overall MAE/RMSE:** average power error over all scored minutes. These need cautious interpretation because inactive minutes dominate the dataset.

I report household-level results and macro averages. A pooled result alone could be dominated by the home with the most usable readings. I also retain the fit and prediction times to show the practical cost of each model.

### 11.1 Metric definitions and macro aggregation

The metric function checks that actual values, predictions and event masks have identical timestamp support. It then calculates regression, event-state, per-minute and energy metrics. Macro aggregation gives each household equal weight.

In [ ]:
def full_utc_grid(index):
    """Return a complete UTC one-minute grid spanning an aware timestamp index."""
    index = pd.DatetimeIndex(index)
    if index.tz is None:
        raise ValueError('LOHO evaluation indices must be timezone-aware.')
    index = index.tz_convert('UTC')
    if len(index) == 0:
        raise ValueError('Cannot build an evaluation grid from an empty index.')
    return pd.date_range(index.min().floor('min'), index.max().floor('min'),
                         freq='1min', tz='UTC')


def predicted_event_state(pred_series, house):
    """Detect predicted events on every timestamp where aggregate-only inference exists.

    Target availability must not determine event segmentation. The supplied series
    therefore contains predictions wherever the model has sufficient aggregate input
    context and NaN only where the aggregate input is unavailable. Scoring is masked
    to common valid target support afterwards.
    """
    full_idx = full_utc_grid(pred_series.index)
    pred_full = pred_series.copy()
    pred_full.index = pd.DatetimeIndex(pred_full.index).tz_convert('UTC')
    pred_full = pred_full.reindex(full_idx)
    events = detect_events(pred_full, house=house)
    state = events_to_binary_series(events, full_idx, include_suspect=True)
    return state, events


def _event_support_fraction(event, support_index):
    support = set(pd.DatetimeIndex(support_index))
    event_idx = pd.date_range(event.start_utc, event.end_utc, freq='1min')
    return (sum(t in support for t in event_idx) / len(event_idx)) if len(event_idx) else 0.0


def count_complete_events_on_support(events, support_index, min_coverage=0.95):
    """Count complete, non-suspect actual events substantially represented in scored support."""
    return sum(
        1 for ev in events
        if (not ev.incomplete and not ev.suspect_fill
            and _event_support_fraction(ev, support_index) >= min_coverage)
    )


def count_predicted_events_on_support(events, support_index, min_coverage=0.95):
    """Count predicted events with adequate scored support; retain suspect predictions for penalties."""
    eligible = [ev for ev in events
                if _event_support_fraction(ev, support_index) >= min_coverage]
    return len(eligible), sum(bool(ev.suspect_fill) for ev in eligible)


def metric_row(y_true, y_pred, true_event_on, pred_event_on, house, model_name,
               n_true_events=0, n_pred_events=0, n_pred_suspect_events=0,
               total_house_minutes=0, valid_target_minutes=0,
               n_train_rows=0, fit_time_s=0.0, predict_time_s=0.0):
    """Calculate all metrics on identical, aligned, valid one-minute observations."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    true_event_on = np.asarray(true_event_on, dtype=bool)
    pred_event_on = np.asarray(pred_event_on, dtype=bool)
    if not (len(y_true) == len(y_pred) == len(true_event_on) == len(pred_event_on)):
        raise ValueError('Metric inputs must have identical lengths and timestamp support.')
    if len(y_true) == 0:
        raise ValueError('Cannot calculate metrics on an empty evaluation support.')
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError('Metric inputs contain non-finite values on scored support.')

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    denom = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = 1.0 - float(np.sum((y_true - y_pred) ** 2)) / denom if denom > 0 else np.nan

    # Event-envelope MAE: use the same 20 W / 27-min ON / 13-min OFF event
    # definition as event-state F1. The envelope includes short, fully observed
    # pauses merged into a cycle; it is not the same as instantaneous >20 W MAE.
    actual_event_minutes = true_event_on
    event_envelope_mae = (
        float(np.mean(np.abs(y_true[actual_event_minutes] - y_pred[actual_event_minutes])))
        if actual_event_minutes.any() else np.nan
    )

    # Event-state F1 uses merged event envelopes. Per-minute F1 uses raw 20 W states.
    ep, er, ef1, _ = precision_recall_fscore_support(
        true_event_on.astype(int), pred_event_on.astype(int),
        average='binary', zero_division=0)
    pp, pr, pf1, _ = precision_recall_fscore_support(
        (y_true > DEFAULT_THRESHOLD_W).astype(int),
        (y_pred > DEFAULT_THRESHOLD_W).astype(int),
        average='binary', zero_division=0)

    # Each scored point is a one-minute mean power measurement. Both series are
    # evaluated on the same aligned support, so sum(W)/60,000 gives comparable kWh.
    e_true = float(y_true.sum() / 60_000.0)
    e_pred = float(y_pred.sum() / 60_000.0)
    e_abs = abs(e_true - e_pred)
    e_rel = e_abs / e_true * 100.0 if e_true > 0 else np.nan

    return {
        'house': int(house), 'model': model_name,
        'n_test_minutes': int(len(y_true)),
        'coverage_pct': round(len(y_true) / max(total_house_minutes, 1) * 100.0, 2),
        'coverage_of_valid_target_pct': round(len(y_true) / max(valid_target_minutes, 1) * 100.0, 2),
        'n_valid_target_minutes_full_grid': int(valid_target_minutes),
        'n_actual_event_envelope_minutes': int(actual_event_minutes.sum()),
        'n_true_event_state_minutes': int(true_event_on.sum()),
        'n_pred_event_state_minutes': int(pred_event_on.sum()),
        'n_true_complete_events_on_support': int(n_true_events),
        'n_pred_events_total': int(n_pred_events),
        'n_pred_events_suspect_over240min': int(n_pred_suspect_events),
        'mae_w': mae, 'rmse_w': rmse, 'r2': r2,
        'event_state_precision': float(ep), 'event_state_recall': float(er),
        'event_state_f1': float(ef1),
        'per_minute_precision': float(pp), 'per_minute_recall': float(pr),
        'per_minute_f1': float(pf1),
        'event_envelope_mae_w': event_envelope_mae,
        'e_actual_kwh': e_true, 'e_pred_kwh': e_pred,
        'e_abs_err_kwh': e_abs, 'e_rel_err_pct': e_rel,
        'n_train_rows': int(n_train_rows),
        'fit_time_s': float(fit_time_s), 'predict_time_s': float(predict_time_s),
    }


def aggregate_macro(results):
    numeric = [
        'mae_w', 'rmse_w', 'r2', 'event_state_precision', 'event_state_recall', 'event_state_f1',
        'per_minute_precision', 'per_minute_recall', 'per_minute_f1', 'event_envelope_mae_w',
        'e_abs_err_kwh', 'e_rel_err_pct', 'coverage_pct', 'coverage_of_valid_target_pct',
        'fit_time_s', 'predict_time_s'
    ]
    rows = []
    for model_name, group in results.groupby('model', sort=False):
        row = {'model': model_name}
        for col in numeric:
            values = pd.to_numeric(group[col], errors='coerce').dropna()
            row[f'{col}_mean'] = values.mean() if len(values) else np.nan
            row[f'{col}_std'] = values.std() if len(values) > 1 else (0.0 if len(values) == 1 else np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


### 11.2 Run the LOHO experiment

Each fold fits on all selected households except one. HistGBR and Seq2Point use the same held-out household; Seq2Point's validation tail comes only from the training households. Predictions are created wherever aggregate context is available, then scored on the common valid-target support.

In [ ]:
def run_loho_comparison(house_features, house_frames, house_meta, metrics_dir=METRICS_DIR):
    """Six-household LOHO benchmark on a common, model-independent target support.

    Event segmentation uses predictions wherever aggregate-only inference is possible,
    not just where target labels happen to be present. This prevents missing target
    labels from artificially breaking predicted events. Metrics are then scored only
    on common valid target timestamps.
    """
    metrics_dir = Path(metrics_dir)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    houses = sorted(house_features)
    if len(houses) < 4:
        raise RuntimeError(f'LOHO requires at least four eligible households; got {len(houses)}.')

    all_rows, full_hgbr_rows, seq_sample_logs, seq_history_logs = [], [], [], []
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Seq2Point device:', device)

    for test_house in houses:
        train_houses = [h for h in houses if h != test_house]
        print(f'\n===== LOHO test=House {test_house}; train={train_houses} =====')
        test_feat = house_features[test_house]
        test_frame = house_frames[test_house]
        test_agg_col, test_target_col = house_meta[test_house]
        full_index = pd.DatetimeIndex(test_frame.index).tz_convert('UTC')
        if not full_index.is_monotonic_increasing or not full_index.is_unique:
            raise AssertionError(f'House {test_house}: expected a unique, sorted one-minute index.')

        # Denominator includes all valid, non-excluded target minutes, even where
        # aggregate input is missing and inference is therefore impossible.
        valid_target_full = test_frame[test_target_col].notna()
        n_valid_target_full = int(valid_target_full.sum())
        if n_valid_target_full == 0:
            raise RuntimeError(f'House {test_house} has no valid target observations.')

        # ----------------------- HistGBR + baselines -----------------------
        sampled_parts = []
        for train_house in train_houses:
            train_h = house_features[train_house].loc[
                house_features[train_house]['target_w'].notna()
            ]
            sampled_parts.append(sample_house(
                train_h, max_rows=MAX_HGBR_ROWS_PER_HOUSE,
                seed=RANDOM_SEED + int(train_house)))
        train_feat = pd.concat(sampled_parts, axis=0)
        n_train_hgbr = len(train_feat)
        hist_model, hist_fit = fit_model(train_feat)
        t0 = time.time()
        pred_hist = pd.Series(predict(hist_model, test_feat), index=test_feat.index, name='HistGBR')
        hist_predict = time.time() - t0

        pred_off = pd.Series(0.0, index=test_feat.index, name='always_off')
        profile = HourOfWeekProfile().fit(train_feat)
        pred_profile = pd.Series(profile.predict(test_feat), index=test_feat.index,
                                 name='hour_of_week_profile')

        # -------------------------- Seq2Point -----------------------------
        train_frames = {h: house_frames[h] for h in train_houses}
        train_meta = {h: house_meta[h] for h in train_houses}
        seq_model, seq_scaler = train_seq2point(train_frames, train_meta, device=device)
        seq_sample_logs.append(seq_scaler['sampling'].assign(test_house=test_house))
        seq_history_logs.append(seq_scaler['history'].assign(test_house=test_house))
        pred_seq, seq_predict_time = predict_seq2point(
            seq_model, seq_scaler, test_frame, test_agg_col, target_col=None)

        # Common score support: complete aggregate-only CNN window AND a valid,
        # non-imputed, non-Issues, non-suspect measured target value.
        common_idx = pred_seq.index.intersection(test_feat.index)
        common_idx = common_idx[test_feat.loc[common_idx, 'target_w'].notna()]
        if len(common_idx) == 0:
            raise RuntimeError(f'House {test_house}: no common valid Seq2Point evaluation timestamps.')

        y_true = test_feat.loc[common_idx, 'target_w'].to_numpy(dtype=float)
        true_state_full = events_to_binary_series(
            eval_events_by_house[test_house], full_index, include_suspect=False)
        true_event_on = true_state_full.reindex(common_idx).fillna(False).to_numpy(dtype=bool)
        actual_events = eval_events_by_house[test_house]
        n_true_common = count_complete_events_on_support(actual_events, common_idx)

        # Build full-grid prediction series using every valid aggregate-only
        # inference timestamp. Target availability is not used for segmentation.
        prediction_series = {
            'always_off': pred_off.reindex(full_index),
            'hour_of_week_profile': pred_profile.reindex(full_index),
            'HistGBR': pred_hist.reindex(full_index),
            'Seq2Point': pred_seq.reindex(full_index),
        }
        predicted_states = {}
        predicted_events = {}
        for model_name, full_pred in prediction_series.items():
            state, events = predicted_event_state(full_pred, test_house)
            predicted_states[model_name] = state.reindex(full_index).fillna(False).astype(bool)
            predicted_events[model_name] = events

        predictions_common = {
            model_name: full_pred.reindex(common_idx).to_numpy(dtype=float)
            for model_name, full_pred in prediction_series.items()
        }
        if any(not np.isfinite(values).all() for values in predictions_common.values()):
            raise AssertionError(f'House {test_house}: non-finite prediction on common support.')

        for model_name, pred_values in predictions_common.items():
            pred_state_common = predicted_states[model_name].reindex(common_idx).to_numpy(dtype=bool)
            pred_event_count, pred_suspect_count = count_predicted_events_on_support(
                predicted_events[model_name], common_idx)
            if model_name == 'Seq2Point':
                n_train = int(seq_scaler['n_train_windows'])
                fit_time = float(seq_scaler['fit_time_s'])
                predict_time = float(seq_predict_time)
            elif model_name == 'HistGBR':
                n_train, fit_time, predict_time = n_train_hgbr, hist_fit, hist_predict
            else:
                n_train, fit_time, predict_time = n_train_hgbr, 0.0, 0.0

            row = metric_row(
                y_true, pred_values, true_event_on, pred_state_common,
                test_house, model_name,
                n_true_events=n_true_common,
                n_pred_events=pred_event_count,
                n_pred_suspect_events=pred_suspect_count,
                total_house_minutes=len(full_index),
                valid_target_minutes=n_valid_target_full,
                n_train_rows=n_train,
                fit_time_s=fit_time,
                predict_time_s=predict_time,
            )
            row['seq2point_best_epoch'] = int(seq_scaler['best_epoch'])
            all_rows.append(row)

        # Supplementary HistGBR/baseline results on every valid target minute
        # for which current aggregate power exists. Not used to compare with CNN.
        full_eval_idx = test_feat.index[test_feat['target_w'].notna()]
        y_full = test_feat.loc[full_eval_idx, 'target_w'].to_numpy(dtype=float)
        true_full = true_state_full.reindex(full_eval_idx).fillna(False).to_numpy(dtype=bool)
        n_true_full = count_complete_events_on_support(actual_events, full_eval_idx)
        for model_name in ['always_off', 'hour_of_week_profile', 'HistGBR']:
            full_pred = prediction_series[model_name]
            pred_full_values = full_pred.reindex(full_eval_idx).to_numpy(dtype=float)
            pred_full_state = predicted_states[model_name].reindex(full_eval_idx).to_numpy(dtype=bool)
            pcount, psuspect = count_predicted_events_on_support(
                predicted_events[model_name], full_eval_idx)
            full_hgbr_rows.append(metric_row(
                y_full, pred_full_values, true_full, pred_full_state,
                test_house, model_name,
                n_true_events=n_true_full,
                n_pred_events=pcount,
                n_pred_suspect_events=psuspect,
                total_house_minutes=len(full_index),
                valid_target_minutes=n_valid_target_full,
                n_train_rows=n_train_hgbr,
                fit_time_s=hist_fit if model_name == 'HistGBR' else 0.0,
                predict_time_s=hist_predict if model_name == 'HistGBR' else 0.0,
            ))

        # Save a complete minute-grid prediction table. Event-state columns use
        # aggregate-only predictions over the full timeline, before score masking.
        out = pd.DataFrame(index=full_index)
        out['y_true'] = test_frame[test_target_col].reindex(full_index)
        out['scored'] = out['y_true'].notna()
        out['common_scored'] = out.index.isin(common_idx)
        out['true_on'] = true_state_full.reindex(full_index).fillna(False).astype(bool)
        out['y_pred_always_off'] = prediction_series['always_off']
        out['y_pred_profile'] = prediction_series['hour_of_week_profile']
        out['y_pred_hgbr'] = prediction_series['HistGBR']
        out['y_pred_seq2point'] = prediction_series['Seq2Point']
        out['seq_scored'] = out['y_pred_seq2point'].notna()
        out['pred_on_histgbr'] = predicted_states['HistGBR']
        out['pred_on_seq2point'] = predicted_states['Seq2Point']
        out.to_parquet(metrics_dir / f'house{test_house}_predictions.parquet')

        print(
            f'Common scored minutes: {len(common_idx):,}; '
            f'valid target minutes on full grid: {n_valid_target_full:,}; '
            f'coverage of valid target: {len(common_idx)/max(n_valid_target_full, 1)*100:.1f}%; '
            f'coverage of full grid: {len(common_idx)/len(full_index)*100:.1f}%'
        )
        del seq_model, hist_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results = pd.DataFrame(all_rows)
    full_results = pd.DataFrame(full_hgbr_rows)
    results.to_csv(metrics_dir / 'loho_results.csv', index=False)
    full_results.to_csv(metrics_dir / 'loho_full_support_results.csv', index=False)
    macro = aggregate_macro(results)
    macro.to_csv(metrics_dir / 'loho_macro.csv', index=False)
    aggregate_macro(full_results).to_csv(metrics_dir / 'loho_full_support_macro.csv', index=False)
    if seq_sample_logs:
        pd.concat(seq_sample_logs, ignore_index=True).to_csv(
            metrics_dir / 'seq2point_training_samples.csv', index=False)
        pd.concat(seq_history_logs, ignore_index=True).to_csv(
            metrics_dir / 'seq2point_training_history.csv', index=False)

    # Final assertions: exactly four model rows per held-out home, common support
    # covers a subset of both full-grid and valid target observations.
    counts = results.groupby('house')['model'].nunique()
    if not counts.eq(4).all() or len(counts) != len(houses):
        raise AssertionError(f'LOHO did not produce four model rows for every house: {counts.to_dict()}')
    if (results['coverage_pct'] > 100.0).any() or (results['coverage_of_valid_target_pct'] > 100.0 + 1e-6).any():
        raise AssertionError('LOHO coverage exceeds 100%; denominator or support is inconsistent.')

    print('\n=== COMMON-SUPPORT LOHO MACRO RESULTS ===')
    display(macro.round(4))
    print('\n=== HISTGBR / BASELINES ON FULL VALID-TARGET SUPPORT ===')
    display(pd.read_csv(metrics_dir / 'loho_full_support_macro.csv').round(4))
    return results, macro, full_results

print('LOHO evaluator defined: full-context event segmentation, valid-target coverage, and aligned metrics.')


## 12. Synthetic checks before using REFIT

Before running the full dataset, I test the two parts of the pipeline where a small logic error could change many downstream results: short-gap interpolation and event segmentation. The synthetic series have known answers, so the checks can verify that leading/trailing gaps remain missing, long gaps are not partially filled, short observed pauses merge correctly, missing gaps split events, and cycle energy is calculated from all valid measured minutes in the event envelope.

These checks test implementation behaviour only. They are not REFIT observations and are not included in any reported performance metric.

In [ ]:
# Synthetic checks: short gaps, protected provider fills, event merging, energy and suspect labels.

def _test_short_gap_interpolation():
    idx = pd.date_range('2020-01-01', periods=12, freq='1min', tz='UTC')
    values = pd.Series([np.nan, 10.0, np.nan, 20.0, np.nan, np.nan,
                       30.0, np.nan, np.nan, np.nan, 40.0, np.nan], index=idx)
    frame = pd.DataFrame({'Aggregate': values,
                          'Aggregate_suspected_fill': [False, False, False, False, True,
                                                       False, False, False, False, False, False, False]})
    out = interpolate_short_gaps(frame, ['Aggregate'], target_col=None, max_gap=2)
    assert pd.isna(out['Aggregate'].iloc[0]) and not out['Aggregate_interpolated'].iloc[0]  # leading gap
    assert out['Aggregate'].iloc[2] == 15.0 and out['Aggregate_interpolated'].iloc[2]       # 1-min internal
    assert out['Aggregate'].iloc[4:6].isna().all() and not out['Aggregate_interpolated'].iloc[4:6].any()  # protected run
    assert out['Aggregate'].iloc[7:10].isna().all() and not out['Aggregate_interpolated'].iloc[7:10].any() # 3-min run
    assert pd.isna(out['Aggregate'].iloc[11]) and not out['Aggregate_interpolated'].iloc[11] # trailing gap

    # The label-only bridge must also respect provider-fill protection.
    protected_target = pd.Series([0.0, 100.0, np.nan, 120.0, 0.0],
                                 index=pd.date_range('2020-02-01', periods=5, freq='1min', tz='UTC'))
    protected = pd.Series([False, False, True, False, False], index=protected_target.index)
    _, bridged, label_imputed = label_events(protected_target, house=98, protected_mask=protected)
    assert pd.isna(bridged.iloc[2]) and not bool(label_imputed.iloc[2])


def _test_cycle_detector():
    idx = pd.date_range('2020-01-01', periods=340, freq='1min', tz='UTC')

    # 20 min ON + 10 valid OFF + 20 min ON => one 50-min envelope; pause energy included.
    p = pd.Series(0.0, index=idx)
    p.iloc[5:25] = 100.0
    p.iloc[25:35] = 10.0
    p.iloc[35:55] = 120.0
    events = detect_events(p, house=99)
    assert len(events) == 1
    ev = events[0]
    assert ev.elapsed_min == 50 and ev.active_min == 40
    expected_kwh = (20*100 + 10*10 + 20*120) / 60_000
    assert abs(ev.energy_kwh - expected_kwh) < 1e-10

    # A 5-minute spike is discarded.
    spike = pd.Series(0.0, index=idx)
    spike.iloc[10:15] = 500.0
    assert len(detect_events(spike, house=99)) == 0

    # A 15-minute valid OFF gap is not merged: both 30-minute events survive separately.
    apart = pd.Series(0.0, index=idx)
    apart.iloc[10:40] = 100.0
    apart.iloc[40:55] = 0.0
    apart.iloc[55:85] = 120.0
    apart_events = detect_events(apart, house=99)
    assert len(apart_events) == 2

    # A missing 5-minute gap is never merged; adjacent events are incomplete.
    gap = pd.Series(0.0, index=idx)
    gap.iloc[10:40] = 100.0
    gap.iloc[40:45] = np.nan
    gap.iloc[45:75] = 120.0
    gap_events = detect_events(gap, house=99)
    assert len(gap_events) == 2 and all(e.incomplete for e in gap_events)

    # Overlong events are retained for audit but excluded from default ground truth.
    long_p = pd.Series(0.0, index=idx)
    long_p.iloc[10:270] = 100.0
    long_events = detect_events(long_p, house=99)
    assert len(long_events) == 1 and long_events[0].suspect_fill
    truth = events_to_binary_series(long_events, idx)
    assert not truth.any()
    pred_state = events_to_binary_series(long_events, idx, include_suspect=True)
    assert pred_state.sum() == 260

_test_short_gap_interpolation()
_test_cycle_detector()
print('Synthetic method checks passed: interpolation, protected fill, cycle merge, missing-gap split, energy and suspect-label handling.')

def _test_metric_active_support_uses_event_envelope():
    # The below-threshold pause is inside the merged event envelope and must be
    # included in event-envelope MAE, while per-minute F1 remains threshold-based.
    y_true = np.array([0.0, 100.0, 10.0, 120.0])
    y_pred = np.array([0.0, 80.0, 0.0, 140.0])
    true_event = np.array([False, True, True, True])
    pred_event = np.array([False, True, True, True])
    row = metric_row(y_true, y_pred, true_event, pred_event, house=97, model_name='synthetic')
    expected = (20.0 + 10.0 + 20.0) / 3.0
    assert abs(row['event_envelope_mae_w'] - expected) < 1e-9
    assert row['n_actual_event_envelope_minutes'] == 3


def _test_hgbr_predict_does_not_require_target():
    class _StubModel:
        def predict(self, X):
            return np.full(len(X), 42.0)
    features_only = pd.DataFrame({'aggregate_current': [100.0, 200.0]})
    pred = predict(_StubModel(), features_only)
    assert np.allclose(pred, [42.0, 42.0])

_test_metric_active_support_uses_event_envelope()
_test_hgbr_predict_does_not_require_target()
print('Additional checks passed: event-envelope MAE support and target-independent inference.')

def _test_invalid_numeric_and_resample_boundary():
    test_idx = pd.date_range('2020-03-01', periods=4, freq='8s', tz='UTC')
    raw = pd.DataFrame({
        'utc': test_idx,
        'Aggregate': [100.0, np.inf, -np.inf, 120.0],
        'Appliance1': [0.0, 20.0, 30.0, 0.0],
    })
    cleaned = flag_invalid_values(raw, ['Aggregate', 'Appliance1'], 'Aggregate')
    assert cleaned['Aggregate'].isna().sum() == 2
    # The last source reading belongs to its floored minute; no ceil-created bin.
    resampled, _, _ = resample_to_1min(cleaned, ['Aggregate', 'Appliance1'])
    assert resampled.index[-1] == test_idx.max().floor('min')

_test_invalid_numeric_and_resample_boundary()
print('Additional checks passed: infinity handling and resampling boundary.')



## 13. Phase 0 — map channels and select households

I first parse the cleaned-data README and verify each mapping against the columns in the corresponding CSV. The preferred cohort is selected by a rule rather than by looking at model results: dedicated `Washing Machine` channel, no documented solar-affected aggregate, no ambiguous channel change, and enough usable data to support EDA and LOHO.

I exclude `Washer Dryer` as a substitute target because it is a different appliance label. The selection table records accepted and rejected homes with a reason. This is important for reproducibility: another person should be able to see why the final cohort contains these households.

In [ ]:
# Parse the official mapping and save it for auditability.
appliance_map, issues_info = parse_readme(README_SOURCE)
appliance_map.to_csv(METRICS_DIR/'appliance_mapping.csv',index=False)

selected_initial = select_households(CLEAN_DIR, appliance_map,
                                     out_csv=METRICS_DIR/'household_selection.csv', target=6)
print('Initial candidate cohort:', selected_initial)
print('\nWashing-machine mappings:')
display(appliance_map[appliance_map.appliance_label.str.lower().str.contains('washing machine|washer dryer',regex=True)])

# Compact manifest from file metadata; timestamps and row counts are verified by the actual cleaning audit later.
def peek_csv(path):
    path=Path(path)
    with path.open('rb') as f:
        header=f.readline().decode(errors='replace').strip().split(',')
        first=f.readline().decode(errors='replace').strip().split(',')
        f.seek(0,2); size=f.tell(); f.seek(max(size-8192,0))
        tail=[x for x in f.read().decode(errors='replace').splitlines() if x.strip()]
        last=tail[-1].split(',') if tail else first
    uidx=next((i for i,c in enumerate(header) if 'unix' in c.lower()),None)
    def to_utc(row):
        if uidx is None or uidx>=len(row): return None
        val=float(row[uidx]); unit='ms' if val>1e11 else 's'
        return pd.to_datetime(val,unit=unit,utc=True)
    return {'file':path.name,'size_mb':round(size/1e6,2),'first_utc':to_utc(first),'last_utc':to_utc(last),'columns':'|'.join(header)}

manifest=[]
sel_df=pd.read_csv(METRICS_DIR/'household_selection.csv')
for h in selected_initial:
    p=find_house_csv(CLEAN_DIR,h)
    if p is not None:
        row=peek_csv(p); row['house']=h; manifest.append(row)
pd.DataFrame(manifest).to_csv(METRICS_DIR/'data_manifest.csv',index=False)
display(pd.read_csv(METRICS_DIR/'household_selection.csv'))
display(pd.DataFrame(manifest).drop(columns=['columns']))

## 14. Phase 1 — raw House 1 audit and cleaning

This is the only section that uses the raw archive. I combine the two House 1 parts, parse timestamps, remove exact duplicates, collapse conflicting timestamps using the per-channel median, flag invalid measurements and resample the readings to one-minute means.

I use a median for conflicting duplicate measurements because it is less sensitive than the mean to an isolated spike among repeated timestamps. Negative appliance readings and raw IAM values above the documented 4,000 W sensor limit are set to missing and logged; large aggregate values are flagged for review rather than deleted automatically.

A minute is retained when at least 50% of the expected source readings are present. I also save coverage under the one-reading and 80% alternatives so the effect of this rule is visible. Only complete internal gaps of one or two minutes are interpolated. Longer outages remain missing and are listed with their start, end and duration.

In [ ]:
raw_house1_files=find_house_csvs(RAW_DIR,1)
if not raw_house1_files:
    raise FileNotFoundError(f'Raw House 1 CSVs not found under {RAW_DIR}. Confirm raw archive extraction.')
print('House 1 raw files:',[p.name for p in raw_house1_files])

df_house1_raw_1min=clean_house(raw_house1_files,house=1,appliance_map=appliance_map,
                               issues_info=issues_info,out_dir=PROC_DIR,metrics_dir=METRICS_DIR,
                               is_raw=True,is_solar=False)
agg_col_h1=df_house1_raw_1min.attrs.get('agg_col') or find_aggregate_col(df_house1_raw_1min)
raw_df_h1=df_house1_raw_1min.attrs['raw_df']
h1_map=appliance_map[appliance_map.house==1]
labels_h1={f'Appliance{int(r.column_index)}':str(r.appliance_label).split(',')[0].strip() for r in h1_map.itertuples()}

def mapped_channel(keyword):
    r=h1_map[h1_map.appliance_label.str.lower().str.contains(keyword)]
    return f"Appliance{int(r['column_index'].iloc[0])}" if len(r) else None

before_after_channels=[c for c in [mapped_channel('washing machine'),mapped_channel('fridge')] if c and c in raw_df_h1.columns]
if len(before_after_channels)<2:
    app_cols=[c for c in raw_df_h1.columns if re.fullmatch(r'appliance\s*\d+',str(c),re.I)]
    before_after_channels=sorted(app_cols,key=lambda c:(-df_house1_raw_1min[c].notna().sum(),c))[:2]

raw_interval=float(raw_df_h1['utc'].sort_values().diff().dt.total_seconds().dropna().median())
plot_daily_coverage(df_house1_raw_1min,[agg_col_h1]+before_after_channels,house=1,
                    out_dir=FIG_DQ_DIR,df_raw=raw_df_h1,raw_interval_s=raw_interval,labels=labels_h1)
flag_times=[]
for fn in ['house1_raw_modified_values_log.csv','house1_raw_duplicates_log.csv']:
    fp=METRICS_DIR/fn
    if fp.exists() and fp.stat().st_size>5:
        try: flag_times += list(pd.to_datetime(pd.read_csv(fp)['utc'],utc=True))
        except Exception: pass
zoom_info=plot_before_after_zoom(raw_df_h1,df_house1_raw_1min,agg_col_h1,before_after_channels,
                                 house=1,out_dir=FIG_DQ_DIR,extra_flag_times=flag_times,labels=labels_h1)

summary_path=METRICS_DIR/'house1_raw_audit_summary.json'
raw_summary=json.loads(summary_path.read_text())
raw_summary['before_after_plot_channels']=[agg_col_h1]+[f'{c} ({labels_h1.get(c,"?")})' for c in before_after_channels]
raw_summary['raw_house1_files']=[p.name for p in raw_house1_files]
raw_summary['zoom_window']=zoom_info
summary_path.write_text(json.dumps(raw_summary,indent=2,default=str))
raw_outages=METRICS_DIR/'house1_raw_outages.csv'
if raw_outages.exists(): shutil.copy2(raw_outages,METRICS_DIR/'outages.csv')

print('House 1 audit:')
display(pd.read_csv(METRICS_DIR/'house1_raw_audit_channels.csv'))
print(json.dumps({k:v for k,v in raw_summary.items() if k!='columns'},indent=2,default=str))
for f in sorted(FIG_DQ_DIR.glob('*.png')):
    print(f.name); display(Image(filename=str(f),width=950))

In [ ]:
from IPython.display import Markdown, display

# Build this interpretation from the audit values produced above.
a = raw_summary
fmt_int = lambda key: f"{int(a[key]):,}" if key in a and pd.notna(a[key]) else "not available"
fmt_num = lambda key, digits=1: f"{float(a[key]):,.{digits}f}" if key in a and pd.notna(a[key]) else "not available"

raw_readout_text = f"""
### What the House 1 audit shows

The raw export contains **{fmt_int('n_rows')} rows** from **{a.get('first_utc', 'not available')}** to **{a.get('last_utc', 'not available')}**. The median observed sampling interval is **{fmt_num('interval_median_s')} seconds**, so I aggregate source readings into one-minute means for the required analysis.

The timestamp audit found **{fmt_int('exact_duplicate_rows')} exact duplicate rows** and **{fmt_int('conflicting_rows_collapsed')} rows at conflicting duplicate timestamps**. I remove exact duplicates and use a per-channel median for conflicting rows. This keeps one record per UTC timestamp while reducing the influence of an isolated extreme value among duplicate measurements.

There are **{fmt_int('intervals_gt_60s')} source intervals longer than one minute** and **{fmt_int('intervals_gt_10min')} longer than ten minutes**. These gaps are why I keep an outage table instead of filling every missing interval. Interpolation is restricted to complete internal runs of one or two one-minute bins; longer outages remain unknown.

The `Time`-column check identifies its interpretation as **{a.get('time_col_interpretation', 'not available')}**. I retain Unix time as the canonical UTC index and use London local time only for calendar summaries. This matters around daylight-saving changes, where a local day can contain 23 or 25 hours.

**How I use these findings:** the audit does not prove that every unusual value is invalid. It tells me which records need to be excluded, flagged or retained with a documented limitation. The before/after plots show how the cleaning rules affect the aggregate and the two mapped appliance channels.
"""
display(Markdown(raw_readout_text))


## 15. Phase 2 — cleaned-house EDA and cycle detection

The official cleaned REFIT files are still sampled at approximately 8-second intervals, so I resample every selected household to one-minute means before calculating EDA statistics. The same coverage rule is used across homes. This matters because cycle counts, average power and missingness would not be comparable if one house were analysed at 8-second resolution and another at one minute.

I use the official README mapping to locate the washing-machine channel in each house. For every home I calculate valid coverage, missingness, the proportion of measured zero readings, power summaries and the number of detected events. I do not treat missing measurements as zero consumption.

The primary event rule is fixed across the cohort: >20 W candidate minutes, merge fully observed OFF pauses shorter than 13 minutes, then retain event envelopes of at least 27 minutes. Q10 is reported only as a sensitivity check. The main household comparisons use the same primary rule so the differences are not caused by changing thresholds between homes.

In [ ]:
selection_df=pd.read_csv(METRICS_DIR/'household_selection.csv')
cohort=selection_df.loc[selection_df.selected,'house'].astype(int).tolist()
spares=sorted(selection_df.loc[selection_df.included & ~selection_df.selected,'house'].astype(int).tolist())
target_house_count=len(cohort)
all_cycles=[]; summaries=[]; checks=[]; final_houses=[]

for house in cohort+spares:
    if len(final_houses)>=target_house_count: break
    csv_path=find_house_csv(CLEAN_DIR,house)
    if csv_path is None:
        checks.append({'house':house,'passed':False,'reason':'cleaned CSV not found'}); continue
    df1=clean_house(csv_path,house=house,appliance_map=appliance_map,issues_info=issues_info,
                    out_dir=PROC_DIR,metrics_dir=METRICS_DIR,is_raw=False,is_solar=(house in SOLAR_HOUSES))
    if df1 is None:
        checks.append({'house':house,'passed':False,'reason':'no dedicated Washing Machine channel'}); continue
    agg_col=find_aggregate_col(df1); target_col=df1.attrs.get('target_col')
    if target_col is None: target_col=find_appliance_col(df1,house,appliance_map,['washing machine'])
    if target_col not in df1.columns:
        checks.append({'house':house,'passed':False,'reason':'mapped target column missing'}); continue

    target_measured=df1[target_col].copy()
    iflag=df1['issue_flag'].fillna(False).astype(bool) if 'issue_flag' in df1.columns else pd.Series(False,index=df1.index)
    joint_ok=df1[agg_col].notna() & target_measured.notna() & ~iflag
    joint_days=float(joint_ok.sum()/1440.0)

    # Shared primary event rule. Bridge only short internal target gaps that are
    # not part of an aggregate-based provider-fill interval.
    protected_fill = (df1.get(f"{agg_col}_suspected_fill", pd.Series(False, index=df1.index))
                      .fillna(False).astype(bool))
    events, bridged_target, imputed_mask = label_events(
        target_measured, house=house, issue_flag=iflag, protected_mask=protected_fill)
    cycles_df=events_to_df(events)
    complete_events=[e for e in events if (not e.incomplete and not e.suspect_fill)]
    passed=(joint_days>=MIN_VALID_DAYS and len(complete_events)>=MIN_COMPLETE_CYCLES)
    checks.append({'house':house,'passed':passed,'joint_valid_days':round(joint_days,1),
                   'detected_events':len(events),'complete_cycles':len(complete_events),
                   'reason':'ok' if passed else f'requires >= {MIN_VALID_DAYS} joint days and >= {MIN_COMPLETE_CYCLES} complete cycles'})
    if not passed:
        # Preserve the evidence but do not include this home's cycles in the final cohort.
        continue

    final_houses.append(house)
    if not cycles_df.empty: all_cycles.append(cycles_df)
    valid=target_measured.dropna()
    wm_label=selection_df.set_index('house').loc[house,'wm_label']
    summary={
        'house':house,'wm_label':wm_label,'target_col':target_col,'agg_col':agg_col,
        'joint_valid_days':round(joint_days,1),'issue_flagged_minutes':int(iflag.sum()),
        'date_range_start':df1.index.min().isoformat(),'date_range_end':df1.index.max().isoformat(),
        'total_minutes':len(df1),'valid_wm_minutes':int(target_measured.notna().sum()),
        'missing_pct':round(target_measured.isna().mean()*100,2),
        'zero_pct':round((valid==0).mean()*100,2) if len(valid) else np.nan,
        'mean_w':round(float(valid.mean()),2) if len(valid) else np.nan,
        'median_w':round(float(valid.median()),2) if len(valid) else np.nan,
        'p95_w':round(float(valid.quantile(.95)),2) if len(valid) else np.nan,
        'max_w':round(float(valid.max()),2) if len(valid) else np.nan,
        'n_detected_events':len(events),'n_complete_cycles':len(complete_events),
        'q10_w':compute_q10_threshold(target_measured),
    }
    summaries.append(summary)
    plot_representative_day(df1,agg_col,target_col,events,house,FIG_EDA_DIR)
    plot_detected_cycles(df1,target_col,events,house,FIG_EDA_DIR)

if len(final_houses)<4:
    raise RuntimeError(f'Only {len(final_houses)} eligible homes after quality checks: {final_houses}. LOHO requires at least four.')

pd.DataFrame(checks).to_csv(METRICS_DIR/'selection_checks.csv',index=False)
summary_df=pd.DataFrame(summaries)
summary_df.to_csv(METRICS_DIR/'household_summary.csv',index=False)
selection_df['phase0_selected']=selection_df['selected']
selection_df['selected']=selection_df.house.isin(final_houses)
selection_df.to_csv(METRICS_DIR/'household_selection.csv',index=False)

if all_cycles:
    cycles_all=pd.concat(all_cycles,ignore_index=True)
    cycles_all.to_csv(METRICS_DIR/'detected_cycles.csv',index=False)
    plot_cross_house_comparison(cycles_all,FIG_EDA_DIR)
    plot_hour_of_day_usage(cycles_all,FIG_EDA_DIR)
    plot_day_of_week_usage(cycles_all,FIG_EDA_DIR)
else:
    cycles_all=pd.DataFrame()

# Consolidate per-house flags/outages generated by clean_house.
for pattern,outname in [('house*_suspected_fill.csv','suspected_fill.csv'),
                        ('house*_outages.csv','outages_cleaned_houses.csv'),
                        ('house*_issue_flags.csv','issue_flags.csv')]:
    parts=[pd.read_csv(p) for p in sorted(METRICS_DIR.glob(pattern)) if p.stat().st_size>5]
    if parts: pd.concat(parts,ignore_index=True).to_csv(METRICS_DIR/outname,index=False)

print('Final selected households:',final_houses)
display(pd.DataFrame(checks))
display(summary_df)
print('Detected cycle summary:')
if not cycles_all.empty:
    display(cycles_all.groupby('house').agg(events=('cycle_id','count'),complete=('incomplete',lambda x:int((~x).sum())),
        median_duration_min=('elapsed_min','median'),median_energy_kwh=('energy_kwh','median')).reset_index())
for f in sorted(FIG_EDA_DIR.glob('*.png')):
    print(f.name); display(Image(filename=str(f),width=950))

In [ ]:
from IPython.display import Markdown, display

# Summarise the household comparison from the Phase 2 tables.
eda_view = summary_df.copy()
eda_view['cycles_per_valid_week'] = eda_view['n_complete_cycles'] / (eda_view['joint_valid_days'] / 7.0)
eda_view = eda_view.sort_values('cycles_per_valid_week', ascending=False)

columns_to_show = [c for c in ['house', 'joint_valid_days', 'n_complete_cycles',
                               'cycles_per_valid_week', 'q10_w', 'missing_pct', 'zero_pct']
                   if c in eda_view.columns]
display(eda_view[columns_to_show].round(2))

if len(eda_view):
    most = eda_view.iloc[0]
    least = eda_view.iloc[-1]
    display(Markdown(f"""
### What the household comparison shows

Under the same 20 W / 27-minute ON / 13-minute OFF rule, **House {int(most['house'])}** has the highest detected complete-cycle rate at **{most['cycles_per_valid_week']:.2f} cycles per valid week**, while **House {int(least['house'])}** has the lowest at **{least['cycles_per_valid_week']:.2f}**. These rates are normalised by valid observed weeks, rather than total calendar duration, so long outages do not make a household appear artificially inactive.

I use one common event rule because the aim here is to compare homes under the same definition. The Q10 value is shown only as a sensitivity check; it does not replace the primary threshold. Cycle counts should be read as events under this operational rule, not as verified customer wash-start records.

Cycle duration and energy describe different aspects of use. Duration measures the event envelope, including short pauses that were merged by the event rule. Energy is calculated from valid measured one-minute mean power inside the event, including those pauses. Events with missing sections or suspect duration are kept visible in the audit but are not treated as complete cycles in the household summaries.
"""))
else:
    display(Markdown('### Household comparison\nNo eligible household summary was produced. Check the selection and data-cleaning outputs before continuing.'))


## 16. Phase 3 — build labels and model features

This step creates two separate things: the regression target and the event labels used for evaluation. The regression target is the measured washing-machine power. Issue-flagged target minutes and all minutes inside suspect-fill event envelopes are excluded from fitting and scoring. The target is never interpolated for regression.

The event labels are built from the measured target with the same event definition used in EDA. Short internal target gaps can be bridged for event continuity only when they are not protected provider-fill intervals; those interpolated minutes are still not scored as measured power. Incomplete but non-suspect events can contribute valid observed minutes to per-minute evaluation, but they are not counted as complete cycles.

The feature set contains aggregate-derived variables only. I keep rows where aggregate input is available even when the target is missing, because a model can still generate an aggregate-only prediction there; the target mask is applied later when calculating training loss and evaluation metrics. This separation prevents target availability from changing the prediction timeline.

In [ ]:
house_features = {}
house_frames = {}
house_meta = {}
model_row_records = []
eval_events_by_house = {}

summary_df = pd.read_csv(METRICS_DIR / 'household_summary.csv').set_index('house')
for house in final_houses:
    p = PROC_DIR / f'house{house}_clean_1min.parquet'
    df1 = pd.read_parquet(p)
    agg_col = summary_df.loc[house, 'agg_col'] if 'agg_col' in summary_df.columns else find_aggregate_col(df1)
    target_col = summary_df.loc[house, 'target_col']
    issue = (df1['issue_flag'].fillna(False).astype(bool) if 'issue_flag' in df1.columns
             else pd.Series(False, index=df1.index))

    # Label from measured target so Issues minutes do not fragment a real event.
    # Protect aggregate-derived provider-fill intervals from label-only interpolation.
    # >240-minute events are quality-control suspects: retain them in the audit
    # table, but mask their entire target interval from training/scoring and reference labels.
    protected_fill = (df1.get(f"{agg_col}_suspected_fill", pd.Series(False, index=df1.index))
                      .fillna(False).astype(bool))
    events, bridged_target, label_imputed = label_events(
        df1[target_col], house=house, issue_flag=issue, protected_mask=protected_fill)
    eval_events_by_house[house] = events
    suspect_mask = event_interval_mask([e for e in events if e.suspect_fill], df1.index)
    true_on = events_to_binary_series(events, df1.index, include_suspect=False)

    df_model = df1.copy()
    target_exclusion = issue | suspect_mask
    df_model.loc[target_exclusion, target_col] = np.nan
    df_model['suspect_event_target_mask'] = suspect_mask
    house_frames[house] = df_model
    house_meta[house] = (agg_col, target_col)

    feat = build_features(df_model, agg_col=agg_col, target_col=target_col)
    feat['true_on'] = true_on.reindex(feat.index).fillna(False).astype(bool)
    house_features[house] = feat

    model_row_records.append({
        'house': house,
        'total_minutes': len(df_model),
        'rows_with_aggregate': len(feat),
        'usable_target_rows': int(feat.target_w.notna().sum()),
        'on_target_minutes_gt20w': int((feat.loc[feat.target_w.notna(), 'target_w'] > DEFAULT_THRESHOLD_W).sum()),
        'true_complete_event_count': sum((not e.incomplete and not e.suspect_fill) for e in events),
        'incomplete_event_count': sum(e.incomplete for e in events),
        'suspect_fill_event_count': sum(e.suspect_fill for e in events),
        'suspect_fill_target_minutes_excluded': int(suspect_mask.sum()),
        'issue_flagged_minutes_excluded': int(issue.sum()),
        'target_missing_pct_after_exclusions': round(df_model[target_col].isna().mean() * 100, 2),
    })
    feat.to_parquet(PROC_DIR / f'house{house}_features.parquet')

model_rows_df = pd.DataFrame(model_row_records)
model_rows_df.to_csv(METRICS_DIR / 'model_rows.csv', index=False)
display(model_rows_df)
print('Ground-truth event labels exclude suspect-fill intervals; issue and suspect target minutes are unscored.')

In [ ]:
from IPython.display import Markdown, display

# Show how much measured target data remains after the quality exclusions.
model_view = model_rows_df.copy()
model_view['active_target_pct'] = (
    model_view['on_target_minutes_gt20w'] / model_view['usable_target_rows'].replace(0, np.nan) * 100
)
display(model_view[['house', 'rows_with_aggregate', 'usable_target_rows',
                     'on_target_minutes_gt20w', 'active_target_pct',
                     'suspect_fill_target_minutes_excluded',
                     'issue_flagged_minutes_excluded']].round(2))

median_active = float(model_view['active_target_pct'].median())
min_rows = int(model_view['usable_target_rows'].min())
max_rows = int(model_view['usable_target_rows'].max())
display(Markdown(f"""
### What this means for model training

Across the selected homes, the median share of usable target minutes above 20 W is **{median_active:.2f}%**. The smallest and largest usable target counts are **{min_rows:,}** and **{max_rows:,}** minutes. This confirms that the regression target is strongly zero-dominated: most training rows describe an inactive washing machine.

I keep the target continuous in watts rather than converting the task into a classifier. The event labels are used for evaluation and for interpreting cycles; they do not replace the measured regression target. The issue and suspect-fill exclusions reduce the amount of labelled data, but they avoid teaching the model that known questionable intervals are valid appliance behaviour.

The input features remain aggregate-only. This is essential to the NILM setup: the whole-house meter is available at inference, while the washing-machine sub-meter exists only to provide the training target and evaluate the estimate.
"""))


## 17. Phase 4 — train and compare the models with LOHO

This is the main modelling experiment. Each fold holds out one household completely. HistGBR is fitted on the training houses' engineered aggregate features. Seq2Point is fitted on aggregate windows from those same training houses; its validation tail is separated in time and used only for early stopping. The held-out house's washing-machine readings are not used until the final metrics are calculated.

The two models are evaluated alongside the always-off and hour-of-week baselines. I compare them on common timestamps, because Seq2Point requires a complete 121-minute aggregate window. The code also records the number of training rows, active target minutes in the training sample, fit time and prediction time.

When reading the table, I do not select a model using overall MAE alone. The always-off baseline can look good simply because the washing machine is usually inactive. Event F1 indicates whether activity periods are found; event-envelope MAE indicates how closely power is estimated during those periods; energy error shows whether the total appliance consumption is preserved. I also check how much each metric varies from one held-out home to another.

In [ ]:
loho_results, loho_macro, loho_full_results = run_loho_comparison(
    house_features=house_features,
    house_frames=house_frames,
    house_meta=house_meta,
    metrics_dir=METRICS_DIR,
)

print('Per-house model results (common support):')
display(loho_results.sort_values(['house','model']).round(4))
print('Macro results (common support):')
display(loho_macro.round(4))

In [ ]:
from IPython.display import Markdown, display

# Compare the two learned models on the same macro-averaged LOHO support.
macro_by_model = loho_macro.set_index('model')
required = ['HistGBR', 'Seq2Point']
missing_models = [m for m in required if m not in macro_by_model.index]
if missing_models:
    raise RuntimeError(f'Missing expected model rows in LOHO results: {missing_models}')

h = macro_by_model.loc['HistGBR']
s = macro_by_model.loc['Seq2Point']

def show_metric(row, name):
    return f"{float(row[name]):.4f}" if name in row.index and pd.notna(row[name]) else 'not available'

f1_winner = 'HistGBR' if h['event_state_f1_mean'] >= s['event_state_f1_mean'] else 'Seq2Point'
energy_winner = 'HistGBR' if h['e_abs_err_kwh_mean'] <= s['e_abs_err_kwh_mean'] else 'Seq2Point'
active_mae_winner = 'HistGBR' if h['event_envelope_mae_w_mean'] <= s['event_envelope_mae_w_mean'] else 'Seq2Point'

display(Markdown(f"""
### How I interpret the LOHO comparison

The macro event-state F1 is **{show_metric(h, 'event_state_f1_mean')} for HistGBR** and **{show_metric(s, 'event_state_f1_mean')} for Seq2Point**. On this metric, **{f1_winner}** is stronger. The event-envelope MAE is **{show_metric(h, 'event_envelope_mae_w_mean')} W** and **{show_metric(s, 'event_envelope_mae_w_mean')} W**, respectively; **{active_mae_winner}** has the lower error on actual event periods. The mean absolute energy errors are **{show_metric(h, 'e_abs_err_kwh_mean')} kWh** and **{show_metric(s, 'e_abs_err_kwh_mean')} kWh**, respectively; **{energy_winner}** is closer on total energy.

I do not combine these measures into one score. Event F1 evaluates whether operating periods are found, event-envelope MAE evaluates power estimation within those periods, and energy error evaluates total consumption over the common scored support. If one architecture leads on one measure and trails on another, that is a real trade-off rather than a reason to hide one metric.

The always-off baseline should also be read carefully. Since the target is zero for most minutes, it may have a lower overall MAE than either learned model while having no useful event-detection capability. This is why I report event F1, event-period error and energy error alongside overall MAE/RMSE.

These are cross-household results for the selected REFIT cohort, not a claim of production-level accuracy across all homes. The per-house table and coverage columns show how much the result varies by held-out household and how much of each target series could actually be scored.
"""))


## 18. Phase 5 — actual-versus-predicted plots and failure analysis

The actual-versus-predicted plot is generated from a held-out household, not a training example. The selection is deterministic: I choose the household closest to the median Seq2Point event-state F1, then select a high-coverage day containing a complete, non-suspect actual cycle whose energy is close to that home's median cycle energy.

The failure table is intended to make the metric differences understandable. A **missed event** is an actual complete, non-suspect event without sufficient overlap from a predicted event. A **false activation** is a predicted event without sufficient overlap from an actual event. I inspect these cases because the same aggregate power level can be produced by different combinations of appliances. A high aggregate load is therefore not enough, by itself, to prove that the washing machine is operating.

### 18.1 Select a held-out day and construct failure cases

The helper functions identify missed actual cycles and false predicted activations from the held-out results. Suspect-fill events and events without sufficient evaluation coverage are excluded so the examples describe model behaviour rather than known data-quality problems.

In [ ]:
def plot_model_day(house, pred_df, df_house, agg_col, out_dir=FIG_MODEL_DIR):
    cycles = [e for e in eval_events_by_house[house] if not e.incomplete and not e.suspect_fill]
    if not cycles:
        print(f'House {house}: no complete non-suspect cycle available for actual-vs-predicted plot.')
        return None
    energies = np.asarray([e.energy_kwh for e in cycles], dtype=float)
    median_energy = float(np.median(energies))
    candidates = []
    target_col = house_meta[house][1]

    for ev in cycles:
        local_day = ev.start_local.normalize()
        day_start = local_day.tz_convert('UTC')
        # Add one local calendar day before converting to UTC. This handles BST
        # transition days (23/25 local hours) correctly.
        day_end = (local_day + pd.DateOffset(days=1)).tz_convert('UTC')
        sl = df_house.loc[(df_house.index >= day_start) & (df_house.index < day_end)]
        if sl.empty:
            continue
        joint = (sl[agg_col].notna() & sl[target_col].notna()).mean()
        candidates.append((float(joint), abs(ev.energy_kwh - median_energy), ev.start_utc,
                           day_start, day_end, ev))

    if not candidates:
        return None
    eligible = [x for x in candidates if x[0] >= 0.95]
    used_fallback = not bool(eligible)
    pool = eligible if eligible else candidates
    chosen = sorted(pool, key=lambda x: (x[1], x[2]))[0]
    joint, _, _, start_utc, end_utc, event = chosen
    sl = pred_df.loc[(pred_df.index >= start_utc) & (pred_df.index < end_utc)]
    if sl.empty:
        return None

    local_idx = sl.index.tz_convert('Europe/London')
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(local_idx, sl.y_true, label='Actual washing-machine power', color='black', lw=1.5)
    ax.plot(local_idx, sl.y_pred_hgbr, label='HistGradientBoosting', lw=1.1, alpha=.85)
    ax.plot(local_idx, sl.y_pred_seq2point, label='Seq2Point CNN', lw=1.1, alpha=.85)
    ax.axhline(DEFAULT_THRESHOLD_W, color='red', ls='--', lw=.8, label='20 W reference threshold')
    ax.axvspan(event.start_local, event.end_local, color='green', alpha=.12, label='Selected complete cycle')
    title = f'House {house} — actual vs predicted — {start_utc.tz_convert("Europe/London").date()}'
    if used_fallback:
        title += ' (highest-coverage fallback; no day reached 95%)'
    ax.set_title(title)
    ax.set_ylabel('Power (W)')
    ax.set_xlabel('Europe/London local time')
    ax.legend()
    ax.grid(alpha=.2)
    path = Path(out_dir) / f'house{house}_actual_vs_predicted.png'
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Selected-day joint coverage:', round(joint * 100, 1), '% | cycle energy:',
          round(event.energy_kwh, 3), 'kWh | fallback:', used_fallback)
    return path


def bool_runs(series):
    s = series.fillna(False).astype(bool)
    full = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq='1min', tz=s.index.tz), fill_value=False)
    starts = full.index[(full & ~full.shift(1, fill_value=False)).to_numpy()]
    ends = full.index[(full & ~full.shift(-1, fill_value=False)).to_numpy()]
    return list(zip(starts, ends))


def make_failure_cases(house, pred_df, pred_col, pred_on_col):
    """Extract interpretable misses/false activations using full event segments.

    Event runs are detected on the full aggregate-only inference timeline. Only
    after segmentation are event intervals intersected with common scored support.
    Events with <95% scored support are omitted from this diagnostic table.
    """
    support = pred_df['common_scored'].fillna(False).astype(bool)
    support_index = pred_df.index[support]
    records = []

    # Missed actual events: only complete, non-suspect actual cycles with at least
    # 95% common evaluation coverage are candidates.
    for ev in eval_events_by_house[house]:
        if ev.incomplete or ev.suspect_fill:
            continue
        event_idx = pd.date_range(ev.start_utc, ev.end_utc, freq='1min')
        event_support = event_idx.isin(support_index)
        if len(event_idx) == 0 or float(event_support.mean()) < 0.95:
            continue
        valid_seg = pred_df.loc[ev.start_utc:ev.end_utc]
        valid_seg = valid_seg[valid_seg['common_scored']]
        if valid_seg.empty:
            continue
        overlap = float(valid_seg[pred_on_col].mean())
        actual_kwh = float(valid_seg['y_true'].sum() / 60_000.0)
        records.append({
            'house': house, 'model': pred_col, 'kind': 'missed_actual_event',
            'start_utc': ev.start_utc, 'end_utc': ev.end_utc,
            'elapsed_min': ev.elapsed_min,
            'common_support_pct': float(event_support.mean() * 100),
            'overlap': overlap, 'actual_kwh': actual_kwh,
            'severity': actual_kwh * (1.0 - overlap),
            'predicted_event_suspect': False,
        })

    # False activations: detect complete predicted runs BEFORE applying the
    # target-support mask. Then retain only events with >=95% scored support.
    pred_state_full = pred_df[pred_on_col].fillna(False).astype(bool)
    for st, en in bool_runs(pred_state_full):
        event_idx = pd.date_range(st, en, freq='1min')
        event_support = event_idx.isin(support_index)
        support_fraction = float(event_support.mean()) if len(event_idx) else 0.0
        if support_fraction < 0.95:
            continue
        valid_seg = pred_df.loc[st:en]
        valid_seg = valid_seg[valid_seg['common_scored']]
        if valid_seg.empty:
            continue
        overlap = float(valid_seg['true_on'].mean())
        pred_kwh = float(valid_seg[pred_col].fillna(0).sum() / 60_000.0)
        elapsed = len(event_idx)
        records.append({
            'house': house, 'model': pred_col, 'kind': 'false_activation',
            'start_utc': st, 'end_utc': en, 'elapsed_min': elapsed,
            'common_support_pct': support_fraction * 100,
            'overlap': overlap, 'predicted_kwh': pred_kwh,
            'severity': pred_kwh * (1.0 - overlap),
            'predicted_event_suspect': elapsed > MAX_CYCLE_DURATION_MIN,
        })
    return records


### 18.2 Generate the actual-versus-predicted and failure figures

The code selects the representative household/day using the fixed rule above, saves the model comparison plots and writes the highest-priority failure examples to CSV.

In [ ]:
# Choose the model with the higher observed macro event-state F1 only to focus
# the illustrative failure-case report. This is a post-evaluation descriptive
# choice, not an independent estimate of generalisation or a production claim.
model_f1 = loho_macro.set_index('model')['event_state_f1_mean']
primary_model = 'Seq2Point' if model_f1.get('Seq2Point', -1) > model_f1.get('HistGBR', -1) else 'HistGBR'
primary_pred_col = 'y_pred_seq2point' if primary_model == 'Seq2Point' else 'y_pred_hgbr'
primary_on_col = 'pred_on_seq2point' if primary_model == 'Seq2Point' else 'pred_on_histgbr'
print('Primary model for failure analysis (highest macro event-state F1):', primary_model)

# Pick the household closest to the median Seq2Point event-state F1; tie-break by ID.
seqrows = loho_results[loho_results.model == 'Seq2Point'].sort_values('house')
median_f1 = float(seqrows.event_state_f1.median())
plot_house = int(seqrows.iloc[int((seqrows.event_state_f1 - median_f1).abs().to_numpy().argmin())].house)
plot_pred = pd.read_parquet(METRICS_DIR / f'house{plot_house}_predictions.parquet')
plot_df = house_frames[plot_house]
plot_path = plot_model_day(plot_house, plot_pred, plot_df, house_meta[plot_house][0])

all_failure = []
for h in final_houses:
    p = pd.read_parquet(METRICS_DIR / f'house{h}_predictions.parquet')
    all_failure += make_failure_cases(h, p, primary_pred_col, primary_on_col)
fail_df = pd.DataFrame(all_failure)
if not fail_df.empty:
    fail_df = fail_df.sort_values('severity', ascending=False)
    fail_df.to_csv(METRICS_DIR / 'failure_cases_all.csv', index=False)
    top = []
    for kind in ['missed_actual_event', 'false_activation']:
        sub = fail_df[fail_df.kind == kind]
        if len(sub):
            top.append(sub.iloc[0])
    top_df = pd.DataFrame(top)
    top_df.to_csv(METRICS_DIR / 'failure_cases.csv', index=False)

    for _, r in top_df.iterrows():
        h = int(r.house)
        p = pd.read_parquet(METRICS_DIR / f'house{h}_predictions.parquet')
        aggregate = house_frames[h][house_meta[h][0]]
        st, en = pd.Timestamp(r.start_utc), pd.Timestamp(r.end_utc)
        lo, hi = st - pd.Timedelta(minutes=90), en + pd.Timedelta(minutes=90)
        s, ag = p.loc[lo:hi], aggregate.loc[lo:hi]
        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        axes[0].plot(ag.index.tz_convert('Europe/London'), ag.values, color='black', lw=1)
        axes[0].set_ylabel('Aggregate (W)')
        axes[0].set_title(f'House {h} — {r.kind} — {st}')
        li = s.index.tz_convert('Europe/London')
        axes[1].plot(li, s.y_true, label='Actual', color='black', lw=1.3)
        axes[1].plot(li, s[primary_pred_col], label=primary_model, lw=1.1)
        axes[1].axhline(DEFAULT_THRESHOLD_W, color='red', ls='--', lw=.8)
        axes[1].axvspan(st.tz_convert('Europe/London'), en.tz_convert('Europe/London'), alpha=.12, color='orange')
        axes[1].set_ylabel('Washing-machine power (W)')
        axes[1].legend()
        axes[1].grid(alpha=.2)
        path = FIG_MODEL_DIR / f'failure_{r.kind}_house{h}.png'
        fig.savefig(path, dpi=160, bbox_inches='tight')
        plt.show()
        plt.close(fig)
    display(top_df)
else:
    top_df = pd.DataFrame()
    top_df.to_csv(METRICS_DIR / 'failure_cases.csv', index=False)
    print('No eligible missed-event or false-activation cases were found on common support.')

# Per-house comparison: event-envelope MAE uses the same duration-filtered
# actual event mask as event-state F1; per-minute F1 uses the instantaneous 20 W rule.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
houses = sorted(loho_results.house.unique())
x = np.arange(len(houses))
width = 0.28
for j, model_name in enumerate(['HistGBR', 'Seq2Point']):
    sub = loho_results[loho_results.model == model_name].set_index('house').reindex(houses)
    axes[0].bar(x + (j - .5) * width, sub.event_envelope_mae_w, width, label=model_name)
    axes[1].bar(x + (j - .5) * width, sub.event_state_f1, width, label=model_name)
axes[0].set_xticks(x); axes[0].set_xticklabels([f'H{h}' for h in houses])
axes[0].set_ylabel('Event-envelope MAE (W)'); axes[0].legend()
axes[0].set_title('Power estimation inside actual event envelopes')
axes[1].set_xticks(x); axes[1].set_xticklabels([f'H{h}' for h in houses])
axes[1].set_ylabel('Event-state F1'); axes[1].set_ylim(0, 1); axes[1].legend()
axes[1].set_title('Per-minute event-envelope agreement')
fig.tight_layout()
fig.savefig(FIG_MODEL_DIR / 'per_house_model_metrics.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)


In [ ]:
from IPython.display import Markdown, display

failure_path = METRICS_DIR / 'failure_cases.csv'
if failure_path.exists() and failure_path.stat().st_size > 10:
    failure_table = pd.read_csv(failure_path)
    display(failure_table)
    display(Markdown(f"""
### Reading the failure cases

The table above is drawn from held-out household predictions. A missed event indicates that the actual signal contains a complete, non-suspect event for which the predicted event does not meet the matching rule. A false activation indicates a predicted event without sufficient overlap with an actual event.

I use these examples to understand the limits of the aggregate signal. Other appliances can create similar power levels, while washing-machine phases can have low or intermittent consumption. A single aggregate snapshot cannot identify the source of a load uniquely; temporal context helps, but it does not remove the ambiguity. The selected cases should therefore be discussed as examples of model error, not as a complete catalogue of every failure mode.
"""))
else:
    display(Markdown('### Failure analysis\nNo failure-case file was produced. Check the Phase 5 diagnostics and the held-out prediction outputs before writing this section of RESULTS.md.'))


## 19. Phase 6 — household usage and illustrative peak windows

This section turns the detected cycles into household-level summaries: cycles per valid week, duration, energy per cycle, weekday/weekend patterns and time-of-day usage.

I calculate observed washing-machine energy in two illustrative local-time windows (16:00–20:00 and 17:00–21:00). These windows are included to show how the analysis could be compared with a utility's actual peak period. They are **not** REFIT tariff periods, measured demand-response opportunities or savings estimates. Without a utility-defined peak window and a customer intervention, I cannot claim that the observed energy is shiftable or that a bill reduction would occur.

In [ ]:
"""
Usage tables (hour-of-day / day-of-week patterns, cycles per week) and the illustrative peak-window energy
for Section 4. Everything is computed from saved outputs (detected_cycles.csv, household_summary.csv and the
processed 1-minute parquet files); nothing here is estimated or assumed except the two scenario windows.
"""
import logging
from pathlib import Path
import pandas as pd

logger = logging.getLogger(__name__)

# Illustrative scenario windows (Europe/London local time, [start, end) hours). NOT tariff periods.
WINDOWS = {"primary_16_20": (16, 20), "sensitivity_17_21": (17, 21)}


def load_complete_cycles(path: Path) -> pd.DataFrame:
    c = pd.read_csv(path)
    c["start_utc"] = pd.to_datetime(c["start_utc"], utc=True)
    c["end_utc"] = pd.to_datetime(c["end_utc"], utc=True)
    c["start_local_ts"] = c["start_utc"].dt.tz_convert("Europe/London")
    c["complete"] = (~c["incomplete"].astype(bool)) & (~c["suspect_fill"].astype(bool))
    return c


def usage_tables(cycles: pd.DataFrame, summary: pd.DataFrame):
    """Per-house usage summary + hour-of-day and day-of-week tables (complete cycles only)."""
    comp = cycles[cycles["complete"]].copy()
    comp["hour"] = comp["start_local_ts"].dt.hour
    comp["dow"] = comp["start_local_ts"].dt.dayofweek          # 0 = Monday
    comp["weekend"] = comp["dow"] >= 5
    s = summary.set_index("house")
    rows = []
    for h, g in comp.groupby("house"):
        weeks = float(s.loc[h, "valid_wm_minutes"]) / 10080.0   # observed time, so outages do not deflate usage
        allc = cycles[cycles["house"] == h]
        rows.append({
            "house": h, "complete_cycles": len(g), "valid_weeks_observed": round(weeks, 1),
            "cycles_per_valid_week": round(len(g) / weeks, 2),
            "duration_mean_min": round(g["elapsed_min"].mean(), 1),
            "duration_median_min": round(g["elapsed_min"].median(), 1),
            "energy_mean_kwh": round(g["energy_kwh"].mean(), 3),
            "energy_median_kwh": round(g["energy_kwh"].median(), 3),
            "peak_w_median": round(g["peak_w"].median(), 0),
            "weekday_cycles": int((~g["weekend"]).sum()), "weekend_cycles": int(g["weekend"].sum()),
            "weekend_share_pct": round(float(g["weekend"].mean() * 100), 1),
            "excluded_incomplete": int(allc["incomplete"].astype(bool).sum()),
            "excluded_suspect_gt240min": int(allc["suspect_fill"].astype(bool).sum()),
        })
    by_hour = (comp.groupby(["house", "hour"]).agg(n_starts=("cycle_id", "size"), energy_kwh=("energy_kwh", "sum"))
               .reset_index())
    by_dow = (comp.groupby(["house", "dow"]).agg(n_starts=("cycle_id", "size"), energy_kwh=("energy_kwh", "sum"))
              .reset_index())
    return pd.DataFrame(rows), by_hour, by_dow


def peak_window_energy(cycles: pd.DataFrame, summary: pd.DataFrame, proc_dir: Path, windows=WINDOWS) -> pd.DataFrame:
    """
    Measured washing-machine kWh that falls inside each illustrative window, counting only minutes inside
    complete cycles. Observational energy use, NOT shiftable energy and NOT a savings estimate.
    """
    s = summary.set_index("house")
    rows = []
    for h, g in cycles[cycles["complete"]].groupby("house"):
        col = s.loc[h, "target_col"]
        tgt = pd.read_parquet(proc_dir / f"house{h}_clean_1min.parquet", columns=[col])[col]
        inside = pd.Series(False, index=tgt.index)
        for st, en in zip(g["start_utc"], g["end_utc"]):
            inside.loc[st:en] = True
        e = tgt.where(inside)                                   # measured power inside complete cycles only
        local_hour = tgt.index.tz_convert("Europe/London").hour
        total = float(e.sum() / 60000.0)
        weeks = float(s.loc[h, "valid_wm_minutes"]) / 10080.0
        row = {"house": h, "complete_cycles": len(g), "wm_kwh_in_complete_cycles": round(total, 2)}
        for name, (a, b) in windows.items():
            kwh = float(e[(local_hour >= a) & (local_hour < b)].sum() / 60000.0)
            starts = g["start_local_ts"].dt.hour
            row[f"energy_in_illustrative_window_kWh_{name}"] = round(kwh, 2)
            row[f"share_of_cycle_energy_{name}_pct"] = round(kwh / total * 100, 1) if total > 0 else None
            row[f"cycle_starts_in_{name}_pct"] = round(float(((starts >= a) & (starts < b)).mean() * 100), 1)
            row[f"kwh_per_valid_week_{name}"] = round(kwh / weeks, 3)
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# Rebuild the cycle table from the current Phase 2 results and add the local-time column expected by usage_tables.
cycles_for_usage=load_complete_cycles(METRICS_DIR/'detected_cycles.csv')
usage_summary,usage_by_hour,usage_by_dow=usage_tables(cycles_for_usage,summary_df.reset_index())
usage_summary.to_csv(METRICS_DIR/'usage_summary.csv',index=False)
usage_by_hour.to_csv(METRICS_DIR/'usage_by_hour.csv',index=False)
usage_by_dow.to_csv(METRICS_DIR/'usage_by_day_of_week.csv',index=False)
peak_df=peak_window_energy(cycles_for_usage,summary_df.reset_index(),PROC_DIR,windows=ILLUSTRATIVE_WINDOWS)
peak_df.to_csv(METRICS_DIR/'peak_window_energy.csv',index=False)

display(usage_summary)
display(peak_df)

# Plot observed cycle-start proportions by local weekday and local hour.
complete=cycles_for_usage[cycles_for_usage.complete].copy()
if len(complete):
    fig,ax=plt.subplots(figsize=(10,4))
    for h,g in complete.groupby('house'):
        hr=g.start_local_ts.dt.hour.value_counts().reindex(range(24),fill_value=0)
        ax.plot(range(24),hr/max(hr.sum(),1),marker='o',ms=3,label=f'House {h}')
    ax.set_xlabel('Local hour'); ax.set_ylabel('Share of cycle starts'); ax.set_title('Washing-machine cycle starts by local hour'); ax.legend(); ax.grid(alpha=.2)
    fig.savefig(FIG_EDA_DIR/'hour_of_day_share_by_house.png',dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)

## 20. Final model comparison and checks

This is the final review of the experiment. I compare model-level macro metrics, inspect the per-house results and verify that every selected household has one result for each of the four approaches. I also check both coverage measures, the common test support and the saved failure-case table.

The result I report should reflect the trade-off actually present in the table. For example, a temporal model may reduce error on active minutes but produce more false activations or larger total energy error. If that happens, I report the trade-off rather than describing the model as better overall. A model that does not improve on a baseline is still a useful finding when the validation is correct and the limitation is explained.

In [ ]:
# Save best/worst households per architecture and print the key metrics.
key_cols=['model','house','coverage_pct','coverage_of_valid_target_pct','mae_w','rmse_w',
          'event_state_precision','event_state_recall','event_state_f1',
          'per_minute_precision','per_minute_recall','per_minute_f1',
          'event_envelope_mae_w','n_actual_event_envelope_minutes','e_abs_err_kwh','e_rel_err_pct']
print('Common-support LOHO per-house results:')
display(loho_results[key_cols].sort_values(['model','house']).round(4))
print('Common-support macro comparison:')
display(loho_macro.round(4))

best_worst=[]
for model_name,g in loho_results.groupby('model'):
    g=g.sort_values('event_state_f1')
    worst_row=g.iloc[0].to_dict(); worst_row['rank']='worst'; best_worst.append(worst_row)
    best_row=g.iloc[-1].to_dict(); best_row['rank']='best'; best_worst.append(best_row)
pd.DataFrame(best_worst).to_csv(METRICS_DIR/'loho_best_worst.csv',index=False)

# Sanity check: all six houses have the same number of model result rows (four models each).
counts=loho_results.groupby('house').model.nunique()
print('Models per test household:',counts.to_dict())
assert (counts==4).all(), 'Every held-out house should have four model rows on common support.'
assert loho_results.groupby('house').coverage_pct.apply(lambda x:x.notna().all()).all()
print('LOHO result sanity checks passed.')

# Consistency checks for the frozen evaluation protocol.
required_metrics = {'event_state_f1', 'per_minute_f1', 'event_envelope_mae_w',
                    'e_abs_err_kwh', 'coverage_pct', 'coverage_of_valid_target_pct'}
assert required_metrics.issubset(set(loho_results.columns)), 'Missing required LOHO metrics.'
assert loho_results.groupby('house').model.nunique().eq(4).all(), 'Each held-out house must have four model rows.'
assert loho_results[['coverage_pct', 'coverage_of_valid_target_pct']].notna().all().all()
assert (loho_results['coverage_pct'] <= 100.0).all()
assert (loho_results['coverage_of_valid_target_pct'] <= 100.0 + 1e-6).all()
print('Frozen LOHO protocol checks passed.')

## 21. Generate the submission documents

The notebook creates `README.md`, `DATA.md`, `RESULTS.md` and `Recommendation.md` from the metric files and summary tables saved by the earlier phases. The source code and notebook together form the fifth deliverable category.

I generate the documents only after the metrics and figures have been produced. This keeps the numerical claims consistent with the executed experiment. The generated text is a starting report based on the outputs; I still read it once for clarity and verify that the selected households, model names, metric definitions and limitations match the final run.

### 21.1 Generate README.md from the executed run

The README records how to reproduce the notebook, what inputs the models use, why the two model families were compared, and which limitations matter for interpretation. It also discloses AI assistance used for code review/documentation support.

In [ ]:
# Generate the five submission documents from executed evidence files only.
raw_audit = json.loads((METRICS_DIR / 'house1_raw_audit_summary.json').read_text())
raw_channels = pd.read_csv(METRICS_DIR / 'house1_raw_audit_channels.csv')
coverage_sensitivity = pd.read_csv(METRICS_DIR / 'house1_raw_coverage.csv')
sel_checks = pd.read_csv(METRICS_DIR / 'selection_checks.csv')
loho_tbl = pd.read_csv(METRICS_DIR / 'loho_macro.csv')
loho_results = pd.read_csv(METRICS_DIR / 'loho_results.csv')
house_tbl = pd.read_csv(METRICS_DIR / 'household_summary.csv')
peak_tbl = pd.read_csv(METRICS_DIR / 'peak_window_energy.csv')


def fmt(v, digits=2):
    try:
        return 'not available' if pd.isna(v) else f'{float(v):,.{digits}f}'
    except Exception:
        return str(v)


def one_row(model):
    rows = loho_tbl[loho_tbl.model == model]
    return rows.iloc[0] if len(rows) else None

house_manifest_cols = ['house', 'target_col', 'date_range_start', 'date_range_end',
                       'joint_valid_days', 'valid_wm_minutes', 'missing_pct']
house_manifest = house_tbl[house_manifest_cols].to_markdown(index=False)

readme_text = f'''# Flock REFIT NILM — WattWise

## Overview
A self-contained Google Colab implementation of the Flock Energy REFIT take-home. All ingestion, cleaning, EDA, model training, evaluation, plotting and report-generation code is embedded in the notebook; no external source ZIP or `src/` package is required.

## Problem
Estimate one-minute washing-machine power from whole-house aggregate power (NILM). The washing-machine channel is the target only; appliance sub-meter channels are not predictive inputs.

## Run
1. Open `flock_refit_final_colab_student_focused.ipynb` in Google Colab.
2. Upload `REFIT_RAW_081116.7z`, `CLEAN_REFIT_081116.7z` and `CLEAN_READ_ME_081116.txt` to `MyDrive/flock-refit-archives/`.
3. Run every cell from top to bottom.
4. Reports, figures and metrics are written under `MyDrive/flock-refit-nilm-final/`; processed Parquet files are stored under `MyDrive/flock-refit-processed/final_v7/`.

## Methods
- Raw House 1: UTC timestamp parsing, exact-duplicate removal, conflicting-timestamp per-channel median, invalid-value audit, one-minute mean resampling, strict internal 1–2 minute gap interpolation and outage logging.
- Cleaned REFIT: parse the official per-house channel map; resample each eligible home from approximately 8-second observations to a complete 1-minute UTC grid; inspect documented `Issues`; flag aggregate-based provider-fill patterns and protect those minutes from being interpolated.
- EDA: six eligible dedicated `Washing Machine` channels; primary 20 W / 27-minute ON / 13-minute OFF event rule; Q10 is only a sensitivity check. Events >240 minutes are retained as suspect-fill candidates but excluded from model labels, training and scoring.
- Models: always-off and training-only hour-of-week baselines; HistGradientBoosting on aggregate-derived lags/rolling statistics; compact Seq2Point CNN on 121-minute aggregate windows.
- Validation: six-fold Leave-One-Household-Out. Seq2Point early stopping uses a disjoint chronological validation tail from the five training houses only. All models are compared on identical common test timestamps.

## Metric definitions
Event-state F1 applies the shared event-duration rule to actual and predicted series, then computes per-minute F1 on event envelopes. Per-minute F1 uses the raw 20 W threshold. Event-envelope MAE is computed on valid target timestamps within the actual event state produced by the shared 20 W / 27-minute ON / 13-minute OFF rule; this includes short pauses merged into an event. Per-minute F1 separately uses the instantaneous >20 W threshold. Energy error is computed on aligned valid one-minute timestamps using `sum(W) / 60,000`. Overall MAE is interpreted cautiously because the target is zero-dominated.

## Dataset attribution and licensing
REFIT is provided by the University of Strathclyde. Retain the dataset attribution and the licence terms stated on the official dataset page (Creative Commons Attribution 4.0 International, where applicable to the supplied release). See the official dataset record: https://pureportal.strath.ac.uk/en/datasets/refit-electrical-load-measurements/.

## Assumptions and trade-offs
The Seq2Point window is centred and therefore suitable for offline retrospective analysis, not causal live inference. A >240-minute event is a quality-control flag, not proof of provider corruption; its target interval is conservatively excluded and reported. The 16:00–20:00 and 17:00–21:00 windows are illustrative local-time windows, not tariff periods or shiftable-energy claims.

## AI tools
AI tools were used for implementation assistance, code review and documentation support. All REFIT statistics, model metrics and plots in this repository are generated by executing the notebook on the supplied files.

## Research references and use
- Murray et al. (2017), REFIT dataset paper, *Scientific Data*, 4:160122 — dataset provenance and measurement context.
- Manca & Massidda (2022), *Communications in Applied and Industrial Mathematics*, 13(1), DOI: 10.2478/caim-2022-0004 — Table 2's study-specific 20 W / 13 min OFF / 27 min ON washing-machine parameters are used for event segmentation.
- Kelly & Knottenbelt (2015), Neural NILM, BuildSys, DOI: 10.1145/2821650.2821672 — background on appliance-state/signature learning.
- Zhang et al. (2018), Seq2Point, AAAI, DOI: 10.1609/aaai.v32i1.11873 — sequence-to-point formulation adapted for the CNN benchmark.

## What I would do next
Collect additional labelled homes with different washing-machine models, test household calibration and domain adaptation, and evaluate whether appliance-specific post-processing improves event detection without sacrificing energy estimation.
'''
(PROJECT_DIR / 'README.md').write_text(readme_text)


### 21.2 Generate DATA.md from the data audit

The data report uses the saved raw audit, coverage and household-manifest files. It describes the preprocessing rules and separates observed data issues from limitations that cannot be recovered from the cleaned release.

In [ ]:
# DATA.md uses observed audit values, not hand-entered estimates.
data_md = f'''# DATA.md — REFIT data audit and preprocessing

## Sources and scope
Raw REFIT House 1 is used for the raw-data cleaning section. Official cleaned REFIT is used for washing-machine EDA and modelling. Appliance-to-column mappings are parsed from the official cleaned README and checked against the CSV columns.

## Raw House 1 findings
- Rows before cleaning: {raw_audit.get('n_rows', 'not available'):,}
- First UTC timestamp: {raw_audit.get('first_utc', 'not available')}
- Last UTC timestamp: {raw_audit.get('last_utc', 'not available')}
- Exact duplicate rows: {raw_audit.get('exact_duplicate_rows', 'not available'):,}
- Conflicting rows collapsed: {raw_audit.get('conflicting_rows_collapsed', 'not available'):,}
- Median source interval: {fmt(raw_audit.get('interval_median_s'))} seconds
- Out-of-order rows: {raw_audit.get('out_of_order_rows', 'not available'):,}
- DST clock-change dates: {', '.join(raw_audit.get('dst_clock_change_days', [])) or 'none detected'}

## Cleaning rules
- Unix timestamp is the canonical UTC time axis; `Europe/London` is used for local hour/day analysis.
- Exact duplicates are dropped. Conflicting rows at one timestamp are collapsed using per-channel median; binary issue flags use the conservative maximum.
- Negative appliance readings are set to missing; negative aggregate readings are flagged and interpreted with solar context. Raw IAM readings above 4,000 W are set to missing. Aggregate values above 23 kW are flagged, not automatically deleted.
- Each selected house is resampled from approximately 8-second readings to a complete one-minute UTC grid using mean power. The primary minute-validity rule is at least 50% of the expected source readings; >=1 and >=80% alternatives are saved as coverage sensitivity results.
- Only complete internal missing runs of one or two minutes are interpolated. Leading/trailing gaps and longer runs remain missing. A gap containing a provider-fill-protected minute is protected in its entirety; it is not partially interpolated.
- The measured washing-machine target is never interpolated for regression training or scoring.
- Aggregate-based provider-fill heuristics are used as quality flags, not proof of an outage. Because the provider may have zero-filled target-only gaps before publication, some target-channel outages cannot be recovered from the cleaned release.
- The documented `Issues` field is used only after its presence and meaning are verified in the supplied README/CSV. Issue-flagged target minutes are excluded from modelling, while issue counts remain available for EDA.

## Final modelling / EDA household manifest
{house_manifest}

## Suspect-fill policy
Events longer than 240 minutes are flagged as suspect-fill candidates. These intervals remain visible in the event audit table but are excluded from model target labels, training and scoring. This is a conservative project quality-control rule, not a universal washing-machine limit.

## Remaining limitations
REFIT aggregate and IAM channels are not perfectly synchronised. The cleaned provider release has already applied some corrections and filling, so not all original outages can be reconstructed. Solar-affected aggregate houses and households with ambiguous or changed washing-machine mappings are excluded from the primary cohort.
'''
(PROJECT_DIR / 'DATA.md').write_text(data_md)


### 21.3 Generate RESULTS.md from EDA and LOHO outputs

The results report includes the selected households, evaluation tables, metric definitions, validation setup and failure analysis. All numerical values are read from the saved CSV/JSON files.

In [ ]:
macro_table = loho_tbl.round(4).to_markdown(index=False)
report_cols = [c for c in [
    'model', 'house', 'coverage_pct', 'coverage_of_valid_target_pct', 'mae_w', 'rmse_w', 'r2',
    'event_state_precision', 'event_state_recall', 'event_state_f1',
    'per_minute_precision', 'per_minute_recall', 'per_minute_f1',
    'event_envelope_mae_w', 'e_abs_err_kwh', 'e_rel_err_pct'
] if c in loho_results.columns]
per_house = loho_results[report_cols].round(4).to_markdown(index=False)
results_md = f'''# RESULTS.md — Washing-machine EDA and disaggregation

## Selected households and periods
{house_manifest}

## Cycle-detection method
A candidate ON minute has measured washing-machine power strictly above 20 W. ON runs separated by fewer than 13 consecutive valid OFF minutes are merged; missing minutes are never bridged. The merged event is retained if its elapsed envelope is at least 27 minutes. This is a study-specific REFIT benchmark rule, not universal appliance physics. Q10 derived from readings >5 W is a secondary sensitivity check only. Events >240 minutes are flagged by a project quality-control heuristic (not a literature threshold) and excluded from complete-cycle summaries, reference activity labels, and supervised target intervals.

## EDA
The notebook generates the required 24-hour aggregate/target trace, detected-cycle plot and cross-household comparison, plus hour-of-day, day-of-week, duration and energy summaries. Each plotted household and date range is listed in the saved manifest. Numerical cycle and household tables are stored under `outputs/metrics/`.

## LOHO model comparison — common evaluation support
All four approaches (always-off, hour-of-week profile, HistGBR and Seq2Point) are compared on identical valid timestamps for each held-out household. Seq2Point requires a complete 121-minute aggregate window. Coverage is reported against the full one-minute household grid and against all valid target minutes on that full grid, including minutes where aggregate input is unavailable. Predicted events are segmented on all aggregate-inference timestamps before scoring is restricted to common valid target support.

{macro_table}

## Per-household results
{per_house}

## Metric interpretation
- Event-state F1 measures per-minute agreement after the shared 20 W / 27-minute ON / 13-minute OFF event post-processing rule.
- Per-minute F1 uses a direct 20 W threshold, without duration post-processing.
- Event-envelope MAE uses scored timestamps inside actual event envelopes defined by the shared 20 W / 27-minute ON / 13-minute OFF rule. It includes short valid pauses merged into a cycle; per-minute F1 separately uses the instantaneous >20 W threshold.
- Energy errors compare actual and predicted power on the same aligned valid one-minute timestamps using `sum(W) / 60,000`.
- The always-off baseline can achieve deceptively low overall MAE because the target is zero-dominated; it should not be considered a useful disaggregation model on that metric alone.

## Validation and training
Six-fold LOHO holds out one complete household at a time. Seq2Point early stopping uses a disjoint chronological validation tail from the five training households only. No held-out household appliance labels enter fitting, scaling or early stopping. The two ML models and baselines are compared on common support. The centred Seq2Point window uses future aggregate context and is therefore an offline/retrospective model; this is a documented inference-mode difference from the causal HistGBR features.

## Failure analysis
The notebook saves the most severe missed complete actual event and false activation for the model with the higher macro event-state F1. Suspect-fill actual intervals and events without sufficient common evaluation support are excluded from missed-cycle analysis.

## Limitations
LOHO across six REFIT homes is useful evidence of cross-household generalisation but does not establish performance across all UK homes, appliance models or India. The Seq2Point model is offline because it uses a centred aggregate window. The event-duration rule is a benchmark segmentation policy and cannot guarantee that each detected event is a complete physical wash cycle.
'''
(PROJECT_DIR / 'RESULTS.md').write_text(results_md)


### 21.4 Generate the one-page utility recommendation

The recommendation reports observed energy in the illustrative windows without converting it into tariff savings. It also describes how a utility pilot could test whether customer recommendations actually change peak-period consumption.

In [ ]:
# Utility-facing one-page recommendation, no invented tariffs/prices/savings.
peak_cols = [c for c in peak_tbl.columns if c.startswith('energy_in_illustrative_window')]
energy_lines = []
for _, row in peak_tbl.iterrows():
    bits = [f"House {int(row['house'])}"]
    for col in peak_cols:
        bits.append(f"{col.split('_')[-1]}: {fmt(row[col], 3)} kWh")
    energy_lines.append('; '.join(bits))
peak_text = '\n'.join('- ' + item for item in energy_lines)
hgbr_row = one_row('HistGBR')
seq_row = one_row('Seq2Point')
rec = f'''# Recommendation.md — Utility-facing summary

WattWise estimates household washing-machine power from whole-house smart-meter readings. It compares two simple baselines, HistGradientBoosting and a temporal Seq2Point CNN using six-fold Leave-One-Household-Out evaluation.

## Findings and suitable use
The observed LOHO comparison is descriptive, not an independent production validation. HistGBR achieved macro event-state F1 {fmt(hgbr_row['event_state_f1_mean'])}, event-envelope MAE {fmt(hgbr_row['event_envelope_mae_w_mean'])} W and absolute energy error {fmt(hgbr_row['e_abs_err_kwh_mean'])} kWh. Seq2Point achieved {fmt(seq_row['event_state_f1_mean'])}, {fmt(seq_row['event_envelope_mae_w_mean'])} W and {fmt(seq_row['e_abs_err_kwh_mean'])} kWh respectively. Event-envelope MAE uses actual event periods defined by the shared 20 W / 27-minute ON / 13-minute OFF rule, including short merged pauses. Neither model should be treated as production-ready; results support exploratory household insights only, not automated billing or guaranteed cycle identification. Overall MAE is interpreted cautiously because an always-off predictor can score deceptively well on a zero-dominated target.

## Potential energy insight
The measured energy summaries below use illustrative local-time windows only. They are not utility tariff periods, shiftable-energy estimates or savings claims. A utility should substitute its actual peak window and determine which cycles customers can safely reschedule.

{peak_text}

## Measuring an intervention
Run a household pilot with a baseline period, recommendation group and control group. Compare peak-window kWh per household, energy shifted, engagement and persistence. REFIT is observational and cannot establish that recommendations cause savings.

## Limitations
Appliance signatures overlap; REFIT covers a limited set of UK households; aggregate and sub-meter sensors are not perfectly synchronised; and provider-filled gaps are not always recoverable. UK washing machines may use internal water heating, whereas many Indian homes use cold-wash or semi-automatic machines. Indian appliance mixes may include large AC and geyser loads, inverters/UPS and voltage variation. Local labelled data is required before deployment in India.

At 15- or 30-minute sampling, cycle starts/ends, brief phases and overlapping appliance transitions become harder to distinguish. Interval energy can remain measurable, but appliance-level temporal detail and cycle detection become less reliable.
'''
if len(rec.split()) > 500:
    raise AssertionError(f'Recommendation exceeds 500 words: {len(rec.split())}')
(PROJECT_DIR / 'Recommendation.md').write_text(rec)

print('Generated documentation:')
for name in ['README.md', 'DATA.md', 'RESULTS.md', 'Recommendation.md']:
    p = PROJECT_DIR / name
    print(name, p.stat().st_size, 'bytes')
print('Recommendation word count:', len(rec.split()))
assert len(rec.split()) <= 500


## 22. Final review before submission

Before publishing the repository, I check that the notebook has completed all phases in order and that the generated files belong to the same run. If I rerun the data cleaning or EDA, I also rerun the dependent feature, modelling, evaluation and reporting phases. I do not combine a new EDA table with old LOHO metrics.

The repository should contain the notebook, the four generated Markdown reports, figures, metric CSVs and `requirements.txt`. I exclude the original REFIT archives, extracted CSVs and processed Parquet files because they are large source data, not project deliverables. The README records the use of AI tools for coding/review assistance; the dataset statistics and model metrics come from executing the notebook on the supplied data.

In [ ]:
print('PROJECT OUTPUTS')
for p in sorted(PROJECT_DIR.rglob('*')):
    if p.is_file():
        print(p.relative_to(PROJECT_DIR), f'{p.stat().st_size/1024:.1f} KB')

print('\nKEY METRIC FILES')
for name in ['household_summary.csv','detected_cycles.csv','model_rows.csv','loho_results.csv',
             'loho_macro.csv','loho_full_support_macro.csv','failure_cases.csv','peak_window_energy.csv']:
    p=METRICS_DIR/name
    print(('OK  ' if p.exists() else 'MISSING  ')+name)

print('\nRequired reports:')
for name in ['README.md','DATA.md','RESULTS.md','Recommendation.md']:
    p=PROJECT_DIR/name
    print(('OK  ' if p.exists() else 'MISSING  ')+name)
print('\nNotebook implementation is self-contained; no source ZIP or src/ directory is required.')

In [ ]:
from IPython.display import Markdown, display

if not peak_df.empty:
    window_col = 'kwh_per_valid_week_primary_16_20'
    if window_col in peak_df.columns:
        peak_sorted = peak_df.sort_values(window_col, ascending=False)
        top = peak_sorted.iloc[0]
        display(Markdown(f"""
### What the usage and peak-window table tells me

Within the illustrative 16:00–20:00 window, **House {int(top['house'])}** has the highest observed complete-cycle energy per valid week at **{float(top[window_col]):.3f} kWh/week**. This is measured energy inside detected complete washing-machine event envelopes, not an estimate of how much energy could be shifted.

I include the 17:00–21:00 window as a sensitivity comparison because the result can change when the assumed analysis window moves. A utility would need to replace both illustrative windows with its own defined peak period, and then test whether customers can actually reschedule cycles. REFIT is observational data, so this analysis alone cannot establish behavioural change or financial savings.
"""))
    else:
        display(Markdown('### Usage interpretation\nThe peak-window output does not contain the expected primary energy column. Inspect `peak_window_energy.csv` before making a numerical statement.'))
else:
    display(Markdown('### Usage interpretation\nNo complete-cycle peak-window summary was produced. Check the detected-cycle table and usage phase.'))
